In [1]:

# ============================================================
# KIDNEYSHIELD — CELL 1
# PubMed clinical-trial and human-evidence retrieval
# ============================================================

import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from IPython.display import display

# ---------- Configuration ----------
BASE_DIR = "/kaggle/working/kidneyshield"
RAW_DIR = os.path.join(BASE_DIR, "raw_data")
OUT_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

EMAIL = ""  # Optional: enter your email for NCBI identification
API_KEY = ""  # Optional: your NCBI API key

CANDIDATES = [
    "pentoxifylline",
    "colchicine",
    "hydroxychloroquine",
    "allopurinol",
    "metformin",
    "pioglitazone",
]

KIDNEY_TERMS = (
    '"chronic kidney disease" OR CKD OR '
    '"diabetic kidney disease" OR "diabetic nephropathy" OR '
    '"renal insufficiency" OR "kidney disease" OR '
    '"kidney function" OR "renal function" OR '
    'eGFR OR proteinuria OR albuminuria'
)

STUDY_TYPES = (
    '"Clinical Trial"[Publication Type] OR '
    '"Randomized Controlled Trial"[Publication Type] OR '
    '"Controlled Clinical Trial"[Publication Type] OR '
    '"Observational Study"[Publication Type] OR '
    '"Systematic Review"[Publication Type] OR '
    '"Meta-Analysis"[Publication Type]'
)

SESSION = requests.Session()
SESSION.headers.update({
    "User-Agent": "KidneyShieldResearch/1.0"
})

def pubmed_request(endpoint, params, retries=3):
    """Make a retryable NCBI E-utilities request."""
    params = dict(params)
    if EMAIL:
        params["email"] = EMAIL
    if API_KEY:
        params["api_key"] = API_KEY

    url = f"https://eutils.ncbi.nlm.nih.gov/entrez/eutils/{endpoint}"

    for attempt in range(retries):
        try:
            response = SESSION.get(
                url, params=params, timeout=45
            )
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            if attempt == retries - 1:
                raise
            print(f"Request retry {attempt + 1}: {exc}")
            time.sleep(2 * (attempt + 1))

def get_text(element, path):
    node = element.find(path)
    return "".join(node.itertext()).strip() if node is not None else ""

def parse_article(article, candidate):
    """Extract fields from a PubMed XML article."""
    pmid = get_text(article, ".//MedlineCitation/PMID")
    title_node = article.find(".//ArticleTitle")
    title = (
        "".join(title_node.itertext()).strip()
        if title_node is not None else ""
    )

    abstract_parts = []
    for node in article.findall(".//Abstract/AbstractText"):
        label = node.attrib.get("Label", "")
        text = "".join(node.itertext()).strip()
        if text:
            abstract_parts.append(
                f"{label}: {text}" if label else text
            )

    abstract = " ".join(abstract_parts)

    journal = get_text(article, ".//Journal/Title")
    year = (
        get_text(article, ".//JournalIssue/PubDate/Year")
        or get_text(article, ".//PubDate/Year")
    )

    if not year:
        medline_date = get_text(article, ".//PubDate/MedlineDate")
        year = medline_date[:4] if medline_date else ""

    doi = ""
    for aid in article.findall(".//PubmedData/ArticleIdList/ArticleId"):
        if aid.attrib.get("IdType", "").lower() == "doi":
            doi = (aid.text or "").strip()
            break

    publication_types = [
        "".join(node.itertext()).strip()
        for node in article.findall(".//PublicationTypeList/PublicationType")
    ]

    authors = []
    for author in article.findall(".//AuthorList/Author"):
        collective = get_text(author, "CollectiveName")
        last = get_text(author, "LastName")
        initials = get_text(author, "Initials")
        name = collective or " ".join(x for x in [last, initials] if x)
        if name:
            authors.append(name)

    return {
        "candidate_drug": candidate,
        "pmid": pmid,
        "title": title,
        "year": year,
        "journal": journal,
        "doi": doi,
        "authors": "; ".join(authors[:6]),
        "publication_types": "; ".join(publication_types),
        "abstract": abstract,
        "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/"
            if pmid else "",
        "retrieval_source": "PubMed E-utilities",
    }

# ---------- Retrieve records ----------
records = []
search_log = []

for drug in CANDIDATES:
    query = (
        f'("{drug}"[Title/Abstract]) AND '
        f'({KIDNEY_TERMS}) AND ({STUDY_TYPES})'
    )

    print(f"\nSearching PubMed: {drug}")

    try:
        response = pubmed_request(
            "esearch.fcgi",
            {
                "db": "pubmed",
                "term": query,
                "retmax": 500,
                "retmode": "json",
                "sort": "relevance",
            },
        )
        search_data = response.json()["esearchresult"]
        pmids = search_data.get("idlist", [])
        count = int(search_data.get("count", 0))

        search_log.append({
            "candidate_drug": drug,
            "pubmed_count_reported": count,
            "pmids_retrieved": len(pmids),
            "query": query,
            "status": "OK",
        })

        print(f"PubMed matches: {count}; retrieved IDs: {len(pmids)}")

        # Fetch in small batches
        for start in range(0, len(pmids), 100):
            batch = pmids[start:start + 100]

            fetch_response = pubmed_request(
                "efetch.fcgi",
                {
                    "db": "pubmed",
                    "id": ",".join(batch),
                    "retmode": "xml",
                },
            )

            root = ET.fromstring(fetch_response.content)

            for article in root.findall(".//PubmedArticle"):
                records.append(parse_article(article, drug))

            time.sleep(0.35)

    except Exception as exc:
        search_log.append({
            "candidate_drug": drug,
            "pubmed_count_reported": "",
            "pmids_retrieved": 0,
            "query": query,
            "status": f"ERROR: {type(exc).__name__}: {exc}",
        })
        print(f"ERROR for {drug}: {exc}")

# ---------- Save raw evidence ----------
evidence = pd.DataFrame(records)
log_df = pd.DataFrame(search_log)

if not evidence.empty:
    evidence["pmid"] = evidence["pmid"].astype(str)
    evidence = evidence.drop_duplicates(
        subset=["pmid", "candidate_drug"]
    )

    evidence.to_csv(
        os.path.join(RAW_DIR, "pubmed_human_evidence.csv"),
        index=False,
    )

    # One PMID can be relevant to multiple candidate searches.
    unique_studies = evidence.drop_duplicates(subset=["pmid"])
    unique_studies.to_csv(
        os.path.join(RAW_DIR, "pubmed_unique_studies.csv"),
        index=False,
    )

log_df.to_csv(
    os.path.join(RAW_DIR, "pubmed_search_log.csv"),
    index=False,
)

print("\n" + "=" * 60)
print("KIDNEYSHIELD PUBMED RETRIEVAL SUMMARY")
print("=" * 60)
print("Drug-search rows:", len(evidence))
print("Unique PMIDs:", evidence["pmid"].nunique() if not evidence.empty else 0)
print("Records with abstracts:",
      int(evidence["abstract"].str.len().gt(0).sum())
      if not evidence.empty else 0)
print("\nSearch log:")
display(log_df)

if not evidence.empty:
    display(
        evidence[
            ["candidate_drug", "pmid", "year", "title",
             "publication_types", "abstract"]
        ].head(15)
    )

print("\nOutput directory:", BASE_DIR)
print("Raw records:", os.path.join(RAW_DIR, "pubmed_human_evidence.csv"))
print("Deduplicated studies:", os.path.join(RAW_DIR, "pubmed_unique_studies.csv"))
print("Search log:", os.path.join(RAW_DIR, "pubmed_search_log.csv"))

if evidence.empty:
    print(
        "\nNo records were retrieved. Check the search log and Kaggle "
        "Internet settings before proceeding."
    )


Searching PubMed: pentoxifylline
PubMed matches: 64; retrieved IDs: 64

Searching PubMed: colchicine
PubMed matches: 31; retrieved IDs: 31

Searching PubMed: hydroxychloroquine
PubMed matches: 46; retrieved IDs: 46

Searching PubMed: allopurinol
PubMed matches: 158; retrieved IDs: 158

Searching PubMed: metformin
PubMed matches: 271; retrieved IDs: 271

Searching PubMed: pioglitazone
PubMed matches: 48; retrieved IDs: 48

KIDNEYSHIELD PUBMED RETRIEVAL SUMMARY
Drug-search rows: 618
Unique PMIDs: 595
Records with abstracts: 614

Search log:


,candidate_drug,pubmed_count_reported,pmids_retrieved,query,status
0,pentoxifylline,64,64,"(""pentoxifylline""[Title/Abstract]) AND (""chron...",OK
1,colchicine,31,31,"(""colchicine""[Title/Abstract]) AND (""chronic k...",OK
2,hydroxychloroquine,46,46,"(""hydroxychloroquine""[Title/Abstract]) AND (""c...",OK
3,allopurinol,158,158,"(""allopurinol""[Title/Abstract]) AND (""chronic ...",OK
4,metformin,271,271,"(""metformin""[Title/Abstract]) AND (""chronic ki...",OK
5,pioglitazone,48,48,"(""pioglitazone""[Title/Abstract]) AND (""chronic...",OK


,candidate_drug,pmid,year,title,publication_types,abstract
0,pentoxifylline,9063959,1996,Pentoxifylline preloading reduces endothelial ...,Clinical Trial; Journal Article; Randomized Co...,"Pentoxifylline (PTX), a methyl xanthine deriva..."
1,pentoxifylline,27849111,2016,Pharmacological interventions for pruritus in ...,Journal Article; Meta-Analysis; Systematic Review,BACKGROUND: This is an update of the original ...
2,pentoxifylline,11378603,2001,Influence of prophylactic use of pentoxifyllin...,Clinical Trial; Journal Article; Randomized Co...,OBJECTIVE: To study the effects of pretreatmen...
3,pentoxifylline,33792832,2021,A randomized double-blinded placebo-controlled...,Journal Article; Multicenter Study; Randomized...,BACKGROUND AND OBJECTIVE: Renal toxicity has l...
4,pentoxifylline,22336824,2012,Pentoxifylline for diabetic kidney disease.,Journal Article; Meta-Analysis; Research Suppo...,BACKGROUND: Diabetic kidney disease (DKD) is a...
5,pentoxifylline,16878429,2006,Effects of pentoxifylline on the urinary prote...,Journal Article; Randomized Controlled Trial,AIMS: The purpose of this study was to identif...
6,pentoxifylline,8981202,1996,Effect of pentoxifylline on renal toxicity of ...,Clinical Trial; Journal Article; Randomized Co...,BACKGROUND: Pentoxifylline was suggested to pr...
7,pentoxifylline,24434385,2014,Evaluation of the effect of pentoxifylline on ...,Clinical Trial; Journal Article,Use of recombinant human erythropoietin (rh-Ep...
8,pentoxifylline,39182114,2024,Pentoxifylline ameliorates subclinical atheros...,Journal Article; Randomized Controlled Trial; ...,BACKGROUND: Diabetic kidney disease (DKD) is a...
9,pentoxifylline,26237421,2015,Pentoxifylline for Anemia in Chronic Kidney Di...,Journal Article; Meta-Analysis; Research Suppo...,BACKGROUND: Pentoxifylline (PTX) is a promisin...



Output directory: /kaggle/working/kidneyshield
Raw records: /kaggle/working/kidneyshield/raw_data/pubmed_human_evidence.csv
Deduplicated studies: /kaggle/working/kidneyshield/raw_data/pubmed_unique_studies.csv
Search log: /kaggle/working/kidneyshield/raw_data/pubmed_search_log.csv


In [2]:

# ============================================================
# KIDNEYSHIELD — CELL 2
# Relevance screening + clinical evidence review queue
# ============================================================

import os
import re
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from IPython.display import display

BASE_DIR = "/kaggle/working/kidneyshield"
RAW_DIR = os.path.join(BASE_DIR, "raw_data")
OUT_DIR = os.path.join(BASE_DIR, "results")

INPUT = os.path.join(RAW_DIR, "pubmed_human_evidence.csv")
PRIORITY_OUT = os.path.join(OUT_DIR, "clinical_evidence_review_queue.csv")
ALL_OUT = os.path.join(OUT_DIR, "all_records_screened.csv")
EXCLUDED_OUT = os.path.join(OUT_DIR, "lower_priority_records.csv")

os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(INPUT):
    raise FileNotFoundError(
        f"Missing input: {INPUT}. Run Cell 1 first."
    )

df = pd.read_csv(INPUT, dtype=str).fillna("")
df["pmid"] = df["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)
df = df.drop_duplicates(subset=["pmid", "candidate_drug"]).copy()

# Add important known studies if not already retrieved by the search.
# These are retrieval targets, not pre-judged positive evidence.
KNOWN_STUDIES = {
    "21144773": "pentoxifylline",
    "22336824": "pentoxifylline",
    "33792832": "pentoxifylline",
    "16878429": "pentoxifylline",
    "35139160": "colchicine",
}

missing = [
    (pmid, drug) for pmid, drug in KNOWN_STUDIES.items()
    if not ((df["pmid"] == pmid) & (df["candidate_drug"] == drug)).any()
]

if missing:
    print("Fetching targeted records absent from Cell 1 results:", missing)
    session = requests.Session()
    for pmid, drug in missing:
        try:
            response = session.get(
                "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
                params={"db": "pubmed", "id": pmid, "retmode": "xml"},
                headers={"User-Agent": "KidneyShieldResearch/1.0"},
                timeout=30,
            )
            response.raise_for_status()
            root = ET.fromstring(response.content)
            articles = root.findall(".//PubmedArticle")

            if not articles:
                print("No PubMed article returned for PMID", pmid)
                continue

            article = articles[0]

            def txt(node, path):
                el = node.find(path)
                return "".join(el.itertext()).strip() if el is not None else ""

            title_node = article.find(".//ArticleTitle")
            title = "".join(title_node.itertext()).strip() if title_node is not None else ""
            abstract = " ".join(
                (
                    (el.attrib.get("Label", "") + ": ")
                    if el.attrib.get("Label") else ""
                ) + "".join(el.itertext()).strip()
                for el in article.findall(".//Abstract/AbstractText")
            ).strip()

            doi = ""
            for aid in article.findall(".//PubmedData/ArticleIdList/ArticleId"):
                if aid.attrib.get("IdType", "").lower() == "doi":
                    doi = (aid.text or "").strip()
                    break

            pub_types = [
                "".join(x.itertext()).strip()
                for x in article.findall(".//PublicationTypeList/PublicationType")
            ]

            year = txt(article, ".//JournalIssue/PubDate/Year") or txt(article, ".//PubDate/Year")
            journal = txt(article, ".//Journal/Title")

            df.loc[len(df)] = {
                "candidate_drug": drug,
                "pmid": pmid,
                "title": title,
                "year": year,
                "journal": journal,
                "doi": doi,
                "authors": "",
                "publication_types": "; ".join(pub_types),
                "abstract": abstract,
                "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
                "retrieval_source": "Targeted PubMed PMID retrieval",
            }
            print("Added PMID", pmid, "-", title[:100])
            time.sleep(0.35)

        except Exception as exc:
            print(f"Could not fetch PMID {pmid}: {exc}")

# ---------- Transparent, rule-based screening ----------
def screen_record(row):
    title = str(row.get("title", "")).lower()
    abstract = str(row.get("abstract", "")).lower()
    ptypes = str(row.get("publication_types", "")).lower()
    text = f"{title} {abstract}"
    drug = str(row.get("candidate_drug", "")).lower()

    score = 0
    reasons = []

    # Kidney and CKD relevance
    chronic_terms = [
        "chronic kidney disease", "ckd", "diabetic kidney disease",
        "diabetic nephropathy", "chronic renal", "renal insufficiency",
        "kidney function", "renal function", "eGFR", "estimated glomerular",
        "proteinuria", "albuminuria", "kidney disease progression",
        "renal progression", "end-stage kidney", "end stage renal",
        "glomerular filtration",
    ]
    chronic_hits = [term for term in chronic_terms if term in text]
    if chronic_hits:
        score += min(6, len(chronic_hits))
        reasons.append("CKD/kidney terms: " + ", ".join(chronic_hits[:5]))

    # Outcomes more directly related to chronic progression
    progression_terms = [
        "eGFR decline", "decline in eGFR", "rate of decline",
        "kidney disease progression", "renal progression",
        "doubling of serum creatinine", "kidney failure",
        "end-stage renal disease", "end stage kidney disease",
        "albuminuria", "proteinuria", "glomerular filtration rate",
    ]
    outcome_hits = [term for term in progression_terms if term in text]
    if outcome_hits:
        score += min(5, len(outcome_hits))
        reasons.append("Potential kidney outcome: " + ", ".join(outcome_hits[:5]))

    # Study type signals
    if "randomized controlled trial" in ptypes or "randomized controlled trial" in text:
        score += 5
        reasons.append("Randomized trial signal")
    elif "randomized" in ptypes or "randomized" in text:
        score += 3
        reasons.append("Randomization mentioned")

    if "clinical trial" in ptypes:
        score += 2
        reasons.append("Clinical trial publication tag")

    if "observational study" in ptypes or "cohort" in text or "case-control" in text:
        score += 2
        reasons.append("Observational design signal")

    if "systematic review" in ptypes or "systematic review" in text:
        score += 3
        reasons.append("Systematic review signal")

    if "meta-analysis" in ptypes or "meta-analysis" in text:
        score += 3
        reasons.append("Meta-analysis signal")

    # Human evidence signals
    if any(x in text for x in [
        "patients were", "participants were", "human subjects",
        "randomly assigned", "randomised", "placebo-controlled",
        "multicenter", "multi-centre",
    ]):
        score += 1
        reasons.append("Human-study signal")

    # Penalize acute-care studies that do not answer chronic progression.
    acute_terms = [
        "contrast-induced nephropathy", "contrast-induced acute kidney injury",
        "contrast media", "renal toxicity of cisplatin",
        "acute kidney injury", "acute renal failure",
        "kidney transplantation", "renal transplantation",
    ]
    acute_hits = [term for term in acute_terms if term in text]
    if acute_hits:
        score -= 5
        reasons.append("Acute/special-context penalty: " + ", ".join(acute_hits[:3]))

    # Penalize records focused on unrelated indications.
    unrelated_terms = [
        "pruritus", "itch", "pharmacokinetics", "hemodialysis anemia",
        "anaemia", "anemia treatment", "erythropoietin",
    ]
    unrelated_hits = [term for term in unrelated_terms if term in text]
    if unrelated_hits and not outcome_hits:
        score -= 3
        reasons.append("Potentially unrelated primary focus: " + ", ".join(unrelated_hits[:3]))

    # Avoid assuming a publication tag proves relevance or efficacy.
    if not str(row.get("abstract", "")).strip():
        score -= 2
        reasons.append("Abstract unavailable")

    if score >= 9:
        tier = "A — high-priority manual review"
    elif score >= 5:
        tier = "B — review"
    elif score >= 2:
        tier = "C — lower priority"
    else:
        tier = "D — likely off-target"

    return pd.Series({
        "screen_score": score,
        "screen_tier": tier,
        "screening_reasons": " | ".join(reasons),
        "manual_verification_required": "YES",
    })

screened_fields = df.apply(screen_record, axis=1)
screened = pd.concat(
    [df.reset_index(drop=True), screened_fields.reset_index(drop=True)],
    axis=1,
)

# Mark the target evidence question explicitly for manual appraisal.
screened["research_question"] = (
    "Does this study provide human evidence relevant to slowing chronic "
    "CKD progression, beyond short-term or surrogate outcomes?"
)

screened = screened.sort_values(
    ["screen_score", "year"],
    ascending=[False, False],
    na_position="last",
)

priority = screened[
    screened["screen_tier"].isin([
        "A — high-priority manual review",
        "B — review",
    ])
].copy()

lower_priority = screened[
    ~screened.index.isin(priority.index)
].copy()

screened.to_csv(ALL_OUT, index=False)
priority.to_csv(PRIORITY_OUT, index=False)
lower_priority.to_csv(EXCLUDED_OUT, index=False)

print("\n" + "=" * 65)
print("KIDNEYSHIELD — SCREENING SUMMARY")
print("=" * 65)
print("Drug-study rows:", len(screened))
print("Unique PMIDs:", screened["pmid"].nunique())
print("\nScreening tiers:")
display(
    screened["screen_tier"]
    .value_counts()
    .rename_axis("tier")
    .reset_index(name="records")
)

print("\nPriority records by drug:")
display(
    priority.groupby("candidate_drug")
    .agg(
        records=("pmid", "size"),
        unique_pmids=("pmid", "nunique"),
        median_screen_score=("screen_score", "median"),
    )
    .reset_index()
    .sort_values("records", ascending=False)
)

print("\nTop 25 records for human review:")
display(
    priority[
        [
            "candidate_drug", "pmid", "year", "title",
            "publication_types", "screen_score",
            "screen_tier", "screening_reasons", "pubmed_url",
        ]
    ].head(25)
)

print("\nSaved outputs:")
print(PRIORITY_OUT)
print(ALL_OUT)
print(EXCLUDED_OUT)

print(
    "\nIMPORTANT: These are heuristic screening labels, not verified "
    "eligibility, risk-of-bias assessments, or efficacy conclusions."
)

Fetching targeted records absent from Cell 1 results: [('35139160', 'colchicine')]
Added PMID 35139160 - Colchicine use and the risk of CKD progression: a multicentre nested case-control study.

KIDNEYSHIELD — SCREENING SUMMARY
Drug-study rows: 619
Unique PMIDs: 596

Screening tiers:


,tier,records
0,B — review,245
1,A — high-priority manual review,223
2,C — lower priority,121
3,D — likely off-target,30



Priority records by drug:


,candidate_drug,records,unique_pmids,median_screen_score
3,metformin,194,194,8.0
0,allopurinol,125,125,8.0
4,pentoxifylline,57,57,9.0
5,pioglitazone,43,43,8.0
2,hydroxychloroquine,25,25,9.0
1,colchicine,24,24,7.0



Top 25 records for human review:


,candidate_drug,pmid,year,title,publication_types,screen_score,screen_tier,screening_reasons,pubmed_url
236,allopurinol,29088122,2017,Xanthine Oxidase Inhibitors for Improving Rena...,Journal Article; Meta-Analysis; Systematic Review,20,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/29088122/
150,allopurinol,40012480,2025,Comparative efficacy and safety of febuxostat ...,Journal Article; Systematic Review; Comparativ...,19,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/40012480/
286,allopurinol,35130544,2022,The Effect of Allopurinol on Renal Outcomes in...,Meta-Analysis; Systematic Review; Journal Article,19,A — high-priority manual review,"CKD/kidney terms: diabetic kidney disease, ren...",https://pubmed.ncbi.nlm.nih.gov/35130544/
10,pentoxifylline,26510426,2016,Effect of pentoxifylline in proteinuric chroni...,Journal Article; Meta-Analysis; Systematic Review,19,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/26510426/
185,allopurinol,24042021,2014,Effects of uric acid-lowering therapy on renal...,Journal Article; Meta-Analysis; Research Suppo...,19,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/24042021/
11,pentoxifylline,18433957,2008,The effect of pentoxifylline on proteinuria in...,Comparative Study; Journal Article; Meta-Analy...,19,A — high-priority manual review,"CKD/kidney terms: diabetic kidney disease, dia...",https://pubmed.ncbi.nlm.nih.gov/18433957/
294,allopurinol,34642880,2022,Prevention of kidney function decline using ur...,Journal Article; Systematic Review; Network Me...,18,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/34642880/
22,pentoxifylline,26995301,2016,Effect of pentoxifylline on renal outcomes in ...,Journal Article; Meta-Analysis; Systematic Review,18,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/26995301/
206,allopurinol,25928556,2015,Urate lowering therapy to improve renal outcom...,Journal Article; Meta-Analysis; Research Suppo...,18,A — high-priority manual review,"CKD/kidney terms: chronic kidney disease, ckd,...",https://pubmed.ncbi.nlm.nih.gov/25928556/
577,pioglitazone,20110146,2010,Effect of thiazolidinediones on albuminuria an...,Journal Article; Meta-Analysis; Systematic Review,17,A — high-priority manual review,"CKD/kidney terms: diabetic nephropathy, protei...",https://pubmed.ncbi.nlm.nih.gov/20110146/



Saved outputs:
/kaggle/working/kidneyshield/results/clinical_evidence_review_queue.csv
/kaggle/working/kidneyshield/results/all_records_screened.csv
/kaggle/working/kidneyshield/results/lower_priority_records.csv

IMPORTANT: These are heuristic screening labels, not verified eligibility, risk-of-bias assessments, or efficacy conclusions.


In [3]:

# ============================================================
# KIDNEYSHIELD — CELL 3
# Focused clinical-trial appraisal dataset
# ============================================================

import os
import re
import pandas as pd
from IPython.display import display

BASE_DIR = "/kaggle/working/kidneyshield"
RAW_DIR = os.path.join(BASE_DIR, "raw_data")
OUT_DIR = os.path.join(BASE_DIR, "results")
os.makedirs(OUT_DIR, exist_ok=True)

INPUT = os.path.join(OUT_DIR, "all_records_screened.csv")

if not os.path.exists(INPUT):
    raise FileNotFoundError(
        f"Cannot find {INPUT}. Run Cells 1 and 2 first."
    )

df = pd.read_csv(INPUT, dtype=str).fillna("")
df["pmid"] = df["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)

# Curated retrieval targets. These are leads to appraise, not
# pre-approved eligible studies or claims of efficacy.
TARGET_PMIDS = {
    "21144773",  # Pentoxifylline trial in diabetic kidney disease
    "22336824",  # Pentoxifylline systematic review
    "33792832",  # Pentoxifylline randomized trial; verify indication/outcome
    "16878429",  # Pentoxifylline and urinary protein
    "30949987",  # Pentoxifylline and CKD progression
    "26510426",  # Pentoxifylline in proteinuric CKD: review
    "28415944",  # Pentoxifylline with ACEI/ARB: proteinuria review
    "35139160",  # Colchicine and CKD progression: nested case-control
    "30922594",  # Hydroxychloroquine in IgA nephropathy
    "29088122",  # Xanthine oxidase inhibitors: renal outcomes review
    "35130544",  # Allopurinol and renal outcomes: review
}

# Clinical progression concepts, separated from surrogate outcomes.
PROGRESSION_TERMS = [
    "ckd progression", "progression of chronic kidney disease",
    "kidney disease progression", "renal progression",
    "eGFR decline", "decline in egfr", "rate of decline",
    "doubling of serum creatinine", "kidney failure",
    "end-stage kidney disease", "end stage renal disease",
    "renal replacement therapy",
]

SURROGATE_TERMS = [
    "proteinuria", "albuminuria", "urinary protein",
    "urine protein", "albumin-to-creatinine",
    "protein-to-creatinine", "glomerular filtration rate",
    "estimated glomerular filtration", "egfr",
]

ACUTE_TERMS = [
    "contrast-induced", "contrast induced", "acute kidney injury",
    "acute renal failure", "cisplatin nephrotoxicity",
    "perioperative", "cardiac surgery",
]

ANIMAL_TERMS = [
    "mouse model", "mice", "rat model", "rats",
    "murine", "in vitro", "cell line",
]

def classify_design(row):
    ptypes = row["publication_types"].lower()
    title = row["title"].lower()
    abstract = row["abstract"].lower()
    text = f"{title} {abstract}"

    if "systematic review" in ptypes or "systematic review" in title:
        return "Systematic review"
    if "meta-analysis" in ptypes or "meta-analysis" in title:
        return "Meta-analysis / evidence synthesis"
    if "randomized controlled trial" in ptypes or "randomized controlled trial" in text:
        return "Randomized controlled trial — verify"
    if "randomized" in text or "randomised" in text:
        return "Possibly randomized — verify"
    if "nested case-control" in text or "case-control" in text:
        return "Case-control / nested case-control — verify"
    if "cohort" in text or "observational study" in ptypes:
        return "Observational cohort — verify"
    if "clinical trial" in ptypes:
        return "Clinical trial — design unclear"
    return "Design unclear — verify"

def classify_outcome(row):
    text = f"{row['title']} {row['abstract']}".lower()

    progression = [x for x in PROGRESSION_TERMS if x in text]
    surrogate = [x for x in SURROGATE_TERMS if x in text]
    acute = [x for x in ACUTE_TERMS if x in text]
    animal = [x for x in ANIMAL_TERMS if x in text]

    if animal:
        context = "Possible preclinical/animal context — verify"
    elif acute:
        context = "Possible acute/special-context kidney outcome — verify"
    elif progression:
        context = "Progression-related outcome mentioned — verify exact endpoint"
    elif surrogate:
        context = "Surrogate kidney outcome mentioned — verify"
    else:
        context = "Kidney outcome not established from title/abstract keywords"

    return pd.Series({
        "outcome_context": context,
        "progression_terms_found": "; ".join(progression),
        "surrogate_terms_found": "; ".join(surrogate),
        "acute_context_terms_found": "; ".join(acute),
    })

# Score title relevance separately from Cell 2's broad keyword score.
def focus_score(row):
    title = row["title"].lower()
    abstract = row["abstract"].lower()
    ptypes = row["publication_types"].lower()
    text = f"{title} {abstract}"

    score = 0

    # Stronger title-level focus
    if any(x in title for x in [
        "ckd progression", "progression of chronic kidney disease",
        "renal outcomes", "kidney outcomes",
    ]):
        score += 5

    if any(x in title for x in [
        "diabetic kidney disease", "diabetic nephropathy",
        "proteinuric chronic kidney disease",
        "igA nephropathy",
    ]):
        score += 3

    if any(x in title for x in [
        "proteinuria", "albuminuria", "renal function",
        "kidney function",
    ]):
        score += 2

    if "randomized controlled trial" in ptypes:
        score += 4
    elif "randomized" in text or "randomised" in text:
        score += 2

    if "systematic review" in ptypes or "meta-analysis" in ptypes:
        score += 2

    if any(x in title for x in ACUTE_TERMS):
        score -= 6

    if any(x in title for x in [
        "pruritus", "pharmacokinetics", "contrast-induced",
        "anemia", "anaemia",
    ]):
        score -= 4

    if not row["abstract"].strip():
        score -= 2

    if row["pmid"] in TARGET_PMIDS:
        score += 8

    return score

df["focus_score"] = df.apply(focus_score, axis=1)
df["provisional_design"] = df.apply(classify_design, axis=1)

outcomes = df.apply(classify_outcome, axis=1)
df = pd.concat([df.reset_index(drop=True), outcomes.reset_index(drop=True)], axis=1)

# Select top 12 per candidate, plus any curated target present in the dataset.
# This prevents metformin/allopurinol's higher search volume from dominating.
selected_parts = []

for drug, group in df.groupby("candidate_drug"):
    group = group.sort_values(
        ["focus_score", "year"],
        ascending=[False, False],
        na_position="last",
    )
    selected_parts.append(group.head(12))

selected = pd.concat(selected_parts, ignore_index=True) if selected_parts else df.head(0)

curated_present = df[df["pmid"].isin(TARGET_PMIDS)]
selected = pd.concat([selected, curated_present], ignore_index=True)
selected = selected.drop_duplicates(subset=["pmid", "candidate_drug"])

# Keep records with an actual abstract/title; flag all for verification.
selected["bibliographic_status"] = "PubMed metadata retrieved; verify citation"
selected["eligibility_status"] = "Not yet adjudicated"
selected["full_text_checked"] = "NO"
selected["human_population_confirmed"] = ""
selected["study_design_confirmed"] = ""
selected["sample_size"] = ""
selected["disease_stage_or_population"] = ""
selected["intervention_dose_duration"] = ""
selected["comparator"] = ""
selected["follow_up_duration"] = ""
selected["primary_kidney_endpoint"] = ""
selected["eGFR_effect_estimate_CI"] = ""
selected["albuminuria_or_proteinuria_effect"] = ""
selected["kidney_failure_or_RRT_result"] = ""
selected["adverse_events"] = ""
selected["risk_of_bias"] = ""
selected["funding_conflicts"] = ""
selected["limitations"] = ""
selected["reviewer_decision"] = ""
selected["reviewer_notes"] = ""

# Use neutral, explicit appraisal questions.
selected["appraisal_question"] = (
    "Does this human study directly inform chronic CKD progression, "
    "or only a surrogate, acute outcome, or unrelated indication?"
)

# Save comprehensive and reviewer-friendly files.
all_screened_out = os.path.join(OUT_DIR, "focused_appraisal_pool.csv")
review_out = os.path.join(OUT_DIR, "manual_study_extraction_template.csv")
summary_out = os.path.join(OUT_DIR, "focused_appraisal_summary.csv")

selected.to_csv(all_screened_out, index=False)

review_columns = [
    "candidate_drug", "pmid", "year", "title", "journal", "doi",
    "pubmed_url", "publication_types", "provisional_design",
    "focus_score", "outcome_context", "progression_terms_found",
    "surrogate_terms_found", "acute_context_terms_found", "abstract",
    "bibliographic_status", "eligibility_status", "full_text_checked",
    "human_population_confirmed", "study_design_confirmed", "sample_size",
    "disease_stage_or_population", "intervention_dose_duration",
    "comparator", "follow_up_duration", "primary_kidney_endpoint",
    "eGFR_effect_estimate_CI", "albuminuria_or_proteinuria_effect",
    "kidney_failure_or_RRT_result", "adverse_events", "risk_of_bias",
    "funding_conflicts", "limitations", "reviewer_decision",
    "reviewer_notes", "appraisal_question",
]

selected[review_columns].to_csv(review_out, index=False)

summary = (
    selected.groupby("candidate_drug")
    .agg(
        selected_records=("pmid", "size"),
        unique_pmids=("pmid", "nunique"),
        trial_tagged=("publication_types",
                      lambda s: s.str.contains("trial", case=False).sum()),
        review_tagged=("publication_types",
                       lambda s: s.str.contains(
                           "systematic review|meta-analysis",
                           case=False, regex=True
                       ).sum()),
        progression_mentions=("progression_terms_found",
                              lambda s: s.str.len().gt(0).sum()),
    )
    .reset_index()
)

summary.to_csv(summary_out, index=False)

print("=" * 65)
print("KIDNEYSHIELD — FOCUSED APPRAISAL DATASET")
print("=" * 65)
print("Selected drug-study rows:", len(selected))
print("Unique PMIDs:", selected["pmid"].nunique())
print("\nPer-drug summary:")
display(summary)

print("\nSelected records for review:")
display(
    selected[
        [
            "candidate_drug", "pmid", "year", "title",
            "provisional_design", "focus_score",
            "outcome_context", "pubmed_url",
        ]
    ]
    .sort_values(["candidate_drug", "focus_score"],
                 ascending=[True, False])
    .head(35)
)

print("\nSaved files:")
print(all_screened_out)
print(review_out)
print(summary_out)

print(
    "\nNOTE: Design and outcome labels are provisional text-based flags. "
    "No study has been confirmed eligible and no efficacy conclusion "
    "has been calculated."
)

KIDNEYSHIELD — FOCUSED APPRAISAL DATASET
Selected drug-study rows: 72
Unique PMIDs: 69

Per-drug summary:


,candidate_drug,selected_records,unique_pmids,trial_tagged,review_tagged,progression_mentions
0,allopurinol,12,12,5,7,7
1,colchicine,12,12,10,1,2
2,hydroxychloroquine,12,12,8,4,1
3,metformin,12,12,9,4,4
4,pentoxifylline,12,12,7,5,6
5,pioglitazone,12,12,9,3,2



Selected records for review:


,candidate_drug,pmid,year,title,provisional_design,focus_score,outcome_context,pubmed_url
0,allopurinol,35130544,2022,The Effect of Allopurinol on Renal Outcomes in...,Systematic review,20,Surrogate kidney outcome mentioned — verify,https://pubmed.ncbi.nlm.nih.gov/35130544/
1,allopurinol,29088122,2017,Xanthine Oxidase Inhibitors for Improving Rena...,Systematic review,14,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/29088122/
2,allopurinol,39551133,2025,Post-hoc analysis of the CARES trial suggests ...,Randomized controlled trial — verify,9,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/39551133/
3,allopurinol,36265753,2022,Baseline risk markers and visit-to-visit varia...,Randomized controlled trial — verify,9,Surrogate kidney outcome mentioned — verify,https://pubmed.ncbi.nlm.nih.gov/36265753/
4,allopurinol,32579811,2020,Effects of Allopurinol on the Progression of C...,Randomized controlled trial — verify,9,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/32579811/
5,allopurinol,32418037,2020,Comparative effect of allopurinol and febuxost...,Systematic review,9,Surrogate kidney outcome mentioned — verify,https://pubmed.ncbi.nlm.nih.gov/32418037/
6,allopurinol,28794291,2017,Allopurinol Against Progression of Chronic Kid...,Randomized controlled trial — verify,9,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/28794291/
7,allopurinol,25928556,2015,Urate lowering therapy to improve renal outcom...,Systematic review,9,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/25928556/
8,allopurinol,24042021,2014,Effects of uric acid-lowering therapy on renal...,Systematic review,9,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/24042021/
9,allopurinol,34642880,2022,Prevention of kidney function decline using ur...,Systematic review,6,Progression-related outcome mentioned — verify...,https://pubmed.ncbi.nlm.nih.gov/34642880/



Saved files:
/kaggle/working/kidneyshield/results/focused_appraisal_pool.csv
/kaggle/working/kidneyshield/results/manual_study_extraction_template.csv
/kaggle/working/kidneyshield/results/focused_appraisal_summary.csv

NOTE: Design and outcome labels are provisional text-based flags. No study has been confirmed eligible and no efficacy conclusion has been calculated.


In [5]:
# Diagnose duplicate column names in the Cell 4 audit DataFrame
print("Duplicate column names:")
print(audit.columns[audit.columns.duplicated(keep=False)].tolist())

print("\nColumns matching acute_context_terms_found:")
print([
    (i, col)
    for i, col in enumerate(audit.columns)
    if col == "acute_context_terms_found"
])

print("\nDataFrame shape:", audit.shape)

Duplicate column names:
['progression_terms_found', 'surrogate_terms_found', 'acute_context_terms_found', 'progression_terms_found', 'surrogate_terms_found', 'acute_context_terms_found']

Columns matching acute_context_terms_found:
[(21, 'acute_context_terms_found'), (50, 'acute_context_terms_found')]

DataFrame shape: (72, 57)


In [6]:
# ============================================================
# CELL 4 FIX: Deduplicate audit columns and rebuild shortlist
# ============================================================

from pathlib import Path
import pandas as pd

OUT = Path("/kaggle/working/kidneyshield/results")
OUT.mkdir(parents=True, exist_ok=True)

# 1. Remove duplicate column names, keeping the first occurrence.
audit = audit.loc[:, ~audit.columns.duplicated(keep="first")].copy()

# 2. Validate required columns.
required = [
    "pmid",
    "candidate_drug",
    "title",
    "abstract",
    "curated_target",
    "acute_context_terms_found",
]

missing = [c for c in required if c not in audit.columns]
if missing:
    raise ValueError(f"Required columns missing from audit: {missing}")

# 3. Robust scalar truth check.
def truthy(value):
    if value is None or pd.isna(value):
        return False
    if isinstance(value, str):
        return value.strip().lower() in {"true", "yes", "1"}
    return bool(value)

# 4. Recalculate priority without relying on ambiguous pandas truth values.
def priority_rank(row):
    score = 0

    if truthy(row.get("curated_target", "")):
        score += 2

    if truthy(row.get("drug_in_title_or_abstract", False)):
        score += 3

    if truthy(row.get("chronic_kidney_terms_found", False)):
        score += 2

    if truthy(row.get("progression_terms_found", False)):
        score += 4

    if truthy(row.get("surrogate_terms_found", False)):
        score += 1

    if truthy(row.get("human_study_design", False)):
        score += 2

    if truthy(row.get("acute_context_terms_found", False)):
        score -= 6

    if truthy(row.get("animal_or_preclinical_terms_found", False)):
        score -= 5

    abstract = row.get("abstract", "")
    if not isinstance(abstract, str) or not abstract.strip():
        score -= 2

    return score

# Use only columns that actually exist; column naming can vary by audit version.
audit["abstract_audit_priority"] = audit.apply(priority_rank, axis=1)

# 5. Sort and deduplicate by PMID, preserving the highest-priority record.
audit = audit.sort_values(
    ["abstract_audit_priority"],
    ascending=False
).copy()

audit["pmid"] = audit["pmid"].astype(str).str.strip()
audit = audit.drop_duplicates(
    subset=["candidate_drug", "pmid"],
    keep="first"
)

# 6. Save the complete audit.
audit_path = OUT / "abstract_eligibility_audit.csv"
audit.to_csv(audit_path, index=False)

# 7. Build a compact review shortlist.
# Keep the highest-priority records per drug for manual assessment.
shortlist = (
    audit.sort_values("abstract_audit_priority", ascending=False)
    .groupby("candidate_drug", group_keys=False)
    .head(8)
    .copy()
)

shortlist_path = OUT / "priority_human_studies.csv"
shortlist.to_csv(shortlist_path, index=False)

# 8. Save lower-priority records for traceability.
priority_pmids = set(shortlist["pmid"])
lower_priority = audit[~audit["pmid"].isin(priority_pmids)].copy()
lower_path = OUT / "off_target_or_unclear_studies.csv"
lower_priority.to_csv(lower_path, index=False)

# 9. Report results.
print("Cell 4 ranking repair completed.")
print("Audit rows:", len(audit))
print("Unique PMIDs:", audit["pmid"].nunique())
print("Shortlist rows:", len(shortlist))
print("\nShortlist by candidate:")
print(shortlist.groupby("candidate_drug").size().to_string())

print("\nDisposition counts:")
if "abstract_level_disposition" in audit.columns:
    print(audit["abstract_level_disposition"].value_counts(dropna=False).to_string())
else:
    print("Disposition column not present.")

print("\nTop 20 for manual review:")
display_cols = [
    c for c in [
        "candidate_drug",
        "pmid",
        "year",
        "title",
        "abstract_level_disposition",
        "abstract_audit_priority",
    ]
    if c in shortlist.columns
]
display(shortlist[display_cols].head(20))

print("\nSaved files:")
print(audit_path)
print(shortlist_path)
print(lower_path)

Cell 4 ranking repair completed.
Audit rows: 72
Unique PMIDs: 69
Shortlist rows: 48

Shortlist by candidate:
candidate_drug
allopurinol           8
colchicine            8
hydroxychloroquine    8
metformin             8
pentoxifylline        8
pioglitazone          8

Disposition counts:
abstract_level_disposition
CKD POPULATION UNCLEAR — manual eligibility check             31
POTENTIAL SURROGATE-ONLY EVIDENCE — verify endpoint           19
POTENTIALLY DIRECT PROGRESSION EVIDENCE — verify full text    17
POTENTIALLY RELEVANT — inspect full text                       4
DRUG ROLE UNCLEAR — manual eligibility check                   1

Top 20 for manual review:


,candidate_drug,pmid,year,title,abstract_level_disposition,abstract_audit_priority
0,allopurinol,35130544,2022,The Effect of Allopurinol on Renal Outcomes in...,POTENTIAL SURROGATE-ONLY EVIDENCE — verify end...,2
1,allopurinol,29088122,2017,Xanthine Oxidase Inhibitors for Improving Rena...,DRUG ROLE UNCLEAR — manual eligibility check,2
53,pentoxifylline,16878429,2006,Effects of pentoxifylline on the urinary prote...,CKD POPULATION UNCLEAR — manual eligibility check,2
52,pentoxifylline,28415944,2017,Pentoxifylline plus ACEIs/ARBs for proteinuria...,POTENTIALLY DIRECT PROGRESSION EVIDENCE — veri...,2
49,pentoxifylline,26510426,2016,Effect of pentoxifylline in proteinuric chroni...,POTENTIAL SURROGATE-ONLY EVIDENCE — verify end...,2
50,pentoxifylline,22336824,2012,Pentoxifylline for diabetic kidney disease.,POTENTIALLY DIRECT PROGRESSION EVIDENCE — veri...,2
24,hydroxychloroquine,30922594,2019,Effects of Hydroxychloroquine on Proteinuria i...,POTENTIALLY DIRECT PROGRESSION EVIDENCE — veri...,2
12,colchicine,35139160,2022,Colchicine use and the risk of CKD progression...,POTENTIALLY DIRECT PROGRESSION EVIDENCE — veri...,2
48,pentoxifylline,30949987,2019,"Pentoxifylline, progression of chronic kidney ...",POTENTIALLY DIRECT PROGRESSION EVIDENCE — veri...,2
51,pentoxifylline,21144773,2011,Pentoxifylline for renoprotection in diabetic ...,POTENTIAL SURROGATE-ONLY EVIDENCE — verify end...,2



Saved files:
/kaggle/working/kidneyshield/results/abstract_eligibility_audit.csv
/kaggle/working/kidneyshield/results/priority_human_studies.csv
/kaggle/working/kidneyshield/results/off_target_or_unclear_studies.csv


In [7]:

# ============================================================
# KIDNEYSHIELD — CELL 5
# Focused PubMed verification + manual evidence extraction
# ============================================================

from pathlib import Path
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from IPython.display import display, Markdown

BASE = Path("/kaggle/working/kidneyshield")
RAW = BASE / "raw_data"
OUT = BASE / "results"
RAW.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

# Focused starting set; this is not an exhaustive systematic review.
TARGETS = {
    "21144773": "pentoxifylline",
    "22336824": "pentoxifylline",
    "35139160": "colchicine",
    "30922594": "hydroxychloroquine",
    "35130544": "allopurinol",
    "29088122": "allopurinol",
    "32579811": "allopurinol",
    "28794291": "allopurinol",
    "25928556": "allopurinol",
    "16878429": "pentoxifylline",
    "28415944": "pentoxifylline",
    "26510426": "pentoxifylline",
    "30949987": "pentoxifylline",
    "33792832": "pentoxifylline",
}

def text_of(parent, path):
    node = parent.find(path)
    if node is None:
        return ""
    return " ".join("".join(node.itertext()).split())

def fetch_pubmed_records(pmids):
    url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
    response = requests.get(
        url,
        params={
            "db": "pubmed",
            "id": ",".join(pmids),
            "retmode": "xml",
        },
        timeout=60,
    )
    response.raise_for_status()
    root = ET.fromstring(response.text)

    records = {}
    for article in root.findall(".//PubmedArticle"):
        pmid = text_of(article, ".//MedlineCitation/PMID")
        if not pmid:
            continue

        title = text_of(article, ".//Article/ArticleTitle")

        abstract_parts = []
        for node in article.findall(".//Article/Abstract/AbstractText"):
            label = node.attrib.get("Label", "")
            value = " ".join("".join(node.itertext()).split())
            if value:
                abstract_parts.append(f"{label}: {value}" if label else value)

        journal = text_of(article, ".//Article/Journal/Title")
        year = (
            text_of(article, ".//Article/Journal/JournalIssue/PubDate/Year")
            or text_of(article, ".//Article/Journal/JournalIssue/PubDate/MedlineDate")
        )

        pub_types = [
            " ".join("".join(n.itertext()).split())
            for n in article.findall(".//Article/PublicationTypeList/PublicationType")
        ]

        authors = []
        for author in article.findall(".//Article/AuthorList/Author"):
            collective = text_of(author, "CollectiveName")
            last = text_of(author, "LastName")
            initials = text_of(author, "Initials")
            name = collective or " ".join(x for x in [last, initials] if x)
            if name:
                authors.append(name)

        records[pmid] = {
            "pmid": pmid,
            "candidate_drug": TARGETS.get(pmid, ""),
            "title": title,
            "year": year,
            "journal": journal,
            "authors": "; ".join(authors[:6]),
            "publication_types": "; ".join(pub_types),
            "abstract": "\n".join(abstract_parts),
            "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
        }

    return records

pmid_list = list(TARGETS.keys())
records = {}

# Batch requests conservatively.
for start in range(0, len(pmid_list), 8):
    batch = pmid_list[start:start + 8]
    try:
        records.update(fetch_pubmed_records(batch))
    except Exception as exc:
        print(f"Fetch error for batch {batch}: {exc}")
    time.sleep(0.4)

# Preserve every requested PMID, including missing records.
rows = []
for pmid, drug in TARGETS.items():
    row = records.get(pmid, {
        "pmid": pmid,
        "candidate_drug": drug,
        "title": "",
        "year": "",
        "journal": "",
        "authors": "",
        "publication_types": "",
        "abstract": "",
        "pubmed_url": f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/",
    })
    rows.append(row)

pubmed_df = pd.DataFrame(rows)
records_path = RAW / "focused_pubmed_verified_records.csv"
pubmed_df.to_csv(records_path, index=False)

print("Records requested:", len(TARGETS))
print("Records returned:", sum(bool(x) for x in pubmed_df["title"]))
print("Records with abstracts:", sum(bool(x) for x in pubmed_df["abstract"]))
print("Saved:", records_path)

# Display full abstracts for direct inspection.
for _, row in pubmed_df.iterrows():
    display(Markdown(
        f"### PMID {row['pmid']} — {row['candidate_drug']}\n"
        f"**{row['title'] or 'Title not retrieved'}**\n\n"
        f"Year: {row['year'] or 'Not retrieved'}  \n"
        f"Journal: {row['journal'] or 'Not retrieved'}  \n"
        f"Publication type: {row['publication_types'] or 'Not retrieved'}  \n"
        f"[Open PubMed record]({row['pubmed_url']})\n\n"
        f"{row['abstract'] or 'No abstract retrieved. Check the PubMed record.'}"
    ))

# Create a blank study-level extraction worksheet.
# Fill from the article/abstract; do not let keyword scores fill these fields.
worksheet = pubmed_df[[
    "pmid", "candidate_drug", "title", "year", "journal",
    "publication_types", "pubmed_url"
]].copy()

manual_fields = {
    "study_design_confirmed": "",
    "drug_is_actual_intervention": "",
    "population_and_disease": "",
    "sample_size": "",
    "baseline_ckd_stage_or_eGFR": "",
    "comparator": "",
    "follow_up_duration": "",
    "primary_endpoint": "",
    "hard_kidney_outcome_reported": "",
    "eGFR_slope_or_change_result": "",
    "albuminuria_or_proteinuria_result": "",
    "effect_estimate_and_95ci": "",
    "statistical_significance": "",
    "adverse_events_and_safety": "",
    "major_bias_or_limitations": "",
    "full_text_checked": "",
    "reviewer_notes": "",
    "evidence_category": "",
}

for col, default in manual_fields.items():
    worksheet[col] = default

worksheet_path = OUT / "manual_study_level_evidence_extraction.csv"
worksheet.to_csv(worksheet_path, index=False)

print("\nManual extraction worksheet saved:", worksheet_path)
display(worksheet.head(10))

Records requested: 14
Records returned: 14
Records with abstracts: 14
Saved: /kaggle/working/kidneyshield/raw_data/focused_pubmed_verified_records.csv


### PMID 21144773 — pentoxifylline
**Pentoxifylline for renoprotection in diabetic nephropathy: the PREDIAN study. Rationale and basal results.**

Year: 2011  
Journal: Journal of diabetes and its complications  
Publication type: Journal Article; Randomized Controlled Trial; Research Support, Non-U.S. Gov't  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/21144773/)

STATEMENTS OF THE PROBLEM: Diabetic nephropathy (DN) is the main cause of end-stage renal disease (ESRD). Renin-angiotensin system (RAS) blockade is the standard of care; however, a significant proportion of patients progress to ESRD. Pentoxifylline (PTF) possesses properties suggesting potential renoprotective efficacy. The aim of the Pentoxifylline for Renoprotection in Diabetic Nephropathy (PREDIAN) study is to test the efficacy of PTF addition to RAS blockade on the progression of DN. Here we report the study design and the baseline patient characteristics.
METHODS: This is an investigator-initiated, single-center, prospective, randomized, controlled, clinical trial without any commercial interest, funded by the Spanish Ministry of Science and Innovation. One hundred and sixty-nine type 2 diabetic patients with Stage 3 and 4 chronic kidney disease (CKD) were randomized to a control group (n=87) or an active group (n=82), which will receive PTF (1200 mg/day) for 24 months. The primary outcome measure is the difference in estimated glomerular filtration rate (eGFR) between the groups at the end of the study.
RESULTS: The baseline characteristics of the subjects are as follows: 116 patients (68.6%) with Stage 3 CKD and 53 (31.3%) Stage 4 CKD, age 69±9 years, duration of diabetes 15±3 years, eGFR 37±12 ml/min per 1.73 m(2), albuminuria 1.39±1.16 g/day, blood pressure 142±8/86±8 mmHg. Inflammatory cytokines (tumor necrosis factor-α, interleukin-6, and interleukin-10) and polymorphisms of the coding genes for these molecules are studied.
CONCLUSIONS: The PREDIAN study will provide evidence on the renoprotective benefit of PTF in addition to interventions of proven efficacy (RAS blockade) in DN.

### PMID 22336824 — pentoxifylline
**Pentoxifylline for diabetic kidney disease.**

Year: 2012  
Journal: The Cochrane database of systematic reviews  
Publication type: Journal Article; Meta-Analysis; Research Support, Non-U.S. Gov't; Systematic Review  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/22336824/)

BACKGROUND: Diabetic kidney disease (DKD) is associated with increased morbidity and mortality, mostly relating to cardiovascular complications. The relevance of inflammation in the pathogenesis of DKD has been investigated in recent years, and it has been shown that inflammatory markers are higher in people with DKD compared with the wider population. Pentoxifylline is a methylxanthine phosphodiesterase inhibitor with favourable anti-inflammatory effects and immunoregulatory properties. The anti-inflammatory effects conferred by pentoxifylline may be beneficial in the management of DKD.
OBJECTIVES: To assess the benefits and harms of pentoxifylline for treating people with DKD.
SEARCH METHODS: We searched the Cochrane Renal Group's specialised register (January 2012), CENTRAL (Issue 12, 2011), MEDLINE, EMBASE and four Chinese biomedical literature databases (CBM-disc, 1979 to July 2009), Chinese Science and Technique Journals Database (VIP, until July 2009), China National Knowledge Infrastructure (CNKI, until July 2009) and WanFang database (until July 2009).
SELECTION CRITERIA: All randomised controlled trials (RCTs) and quasi-RCTs studying the benefits and harms of pentoxifylline for DKD.
DATA COLLECTION AND ANALYSIS: Data were extracted independently by two authors. Meta-analyses were performed when more than one study provided data on a comparable outcome in sufficiently similar patients. Results of dichotomous outcomes were expressed as risk ratios (RR) with 95% confidence intervals (CI). Mean differences (MD) were calculated to assess the effects of treatment where outcomes were expressed on continuous scales, and standardised mean differences (SMD) calculated where different scales were used. Data was pooled using the random effects model. Adverse effects were assessed using descriptive techniques and where possible, risk differences (RD) with 95% CI.
MAIN RESULTS: We identified 17 studies that included a total of 991 participants with DKD which met our inclusion criteria. Overall, the methodological quality of included studies was low: 4/17 reported the method of randomisation, 13/17 did not; no study described the method of random allocation; 4/17 studies were considered to be at high risk of bias and 13/17 were considered to have unclear risk for incomplete outcome data reporting; 9/17 studies were at low risk bias and in 8/17 the risk of bias was unclear for selective outcome reporting.Compared with placebo, pentoxifylline significantly reduced serum creatinine (SCr) (MD -0.10 mg/dL, 95% CI -0.17 to -0.03), albuminuria (SMD -2.28, 95% CI -3.85 to -0.70) and overt proteinuria (MD -428.58 µg/min, 95% CI -661.65 to -195.50), but there was no difference in creatinine clearance (CrCl) (MD -5.18 mL/min, 95% CI -15.55 to 5.19). When compared with routine treatment alone, pentoxifylline did not significantly reduce SCr (MD 0.00 mg/dL, 95% CI -0.06 to 0.07) or blood pressure (systolic (SBP): MD -0.28 mm Hg, 95% CI -2.20 to 1.63; diastolic (DBP): MD -0.15 mm Hg, 95% CI -1.44 to 1.14), but did significantly reduce albuminuria (SMD 0.62, 95% CI 0.18 to 1.07) and proteinuria (MD 0.46 g/24 h, 95% CI 0.17 to 0.74). There was no significant difference in SCr (MD 0.00 mg/dL, 95% CI -0.08 to 0.07), albuminuria (MD -8.79 µg/min, 95% CI -27.18 to 9.59), proteinuria (MD -0.01 g/24 h, 95% CI -0.03 to 0.01) or blood pressure (SBP: MD 1.46 mm Hg, 95% CI -0.57 to 3.50; DBP: MD 1.37 mm Hg, 95% CI -0.23 to 2.98) between pentoxifylline and the active comparator (captopril or clonidine/methyldopa) for patients with type 1 and type 2 DKD. CrCl was significantly increased when pentoxifylline was compared to clonidine/methyldopa (MD 10.90 mL/min, 95% CI -1.40 to 20.40) but not with captopril (MD 3.26 mL/min, 95% CI -1.05 to 7.59). No data were available on the incidence of end-stage kidney disease (ESKD), time to ESKD, quality of life, or all-cause mortality. The adverse events of pentoxifylline were mild; no serious adverse events were reported in any of the included studies.
AUTHORS' CONCLUSIONS: From the available evidence, pentoxifylline seems to offer some beneficial effects in renal function improvement and reduction in albuminuria and proteinuria, with no obvious serious adverse effects for patients with DKD. However, most studies were poorly reported, small, and methodologically flawed. Evidence to support the use of pentoxifylline for DKD was insufficient to develop recommendations for its use in this patient population. Rigorously designed, randomised, multicentre, large scale studies of pentoxifylline for DKD are needed to further assess its therapeutic effects.

### PMID 35139160 — colchicine
**Colchicine use and the risk of CKD progression: a multicentre nested case-control study.**

Year: 2022  
Journal: Rheumatology (Oxford, England)  
Publication type: Multicenter Study; Journal Article  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/35139160/)

OBJECTIVES: Despite the preclinical evidence on protective effects of colchicine against kidney fibrosis, whether colchicine could delay the progression of chronic kidney disease (CKD) in humans remains unknown. This study examined the association between long-term colchicine use and risk of adverse kidney outcome in patients with CKD who were treated for hyperuricaemia or chronic gout.
METHODS: We conducted a multicentre, nested, case-control study in three Korean hospitals. Patients were aged ≥19 years; had CKD G3-G4; and used drugs including colchicine, allopurinol and febuxostat for hyperuricaemia or chronic gout during the period from April 2000 to October 2020. Patients with CKD progression, which was defined as ≥40% decrease from the baseline estimated glomerular filtration rate or the onset of kidney failure with replacement therapy, were matched to controls based on follow-up time, age and sex.
RESULTS: Overall, 3085 patients with CKD progression were matched to 11 715 control patients. Multivariate conditional logistic regression analysis showed that patients with ≥90 cumulative daily colchicine doses were associated with a lower risk of CKD progression [adjusted odds ratio (AOR), 0.77; 95% CI: 0.61, 0.96] than non-users. In the sensitivity analysis with matched CKD stages, the AOR was 0.77 (95% CI: 0.62, 0.97). This association was more pronounced in patients without diabetes or hypertension, and in patients with CKD G3.
CONCLUSION: Colchicine use is associated with a lower risk of adverse kidney outcomes in CKD patients with hyperuricaemia, or chronic gout.

### PMID 30922594 — hydroxychloroquine
**Effects of Hydroxychloroquine on Proteinuria in IgA Nephropathy: A Randomized Controlled Trial.**

Year: 2019  
Journal: American journal of kidney diseases : the official journal of the National Kidney Foundation  
Publication type: Clinical Trial, Phase II; Journal Article; Randomized Controlled Trial; Research Support, Non-U.S. Gov't  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/30922594/)

RATIONALE & OBJECTIVE: Despite optimization of renin-angiotensin-aldosterone system (RAAS) inhibition, patients with immunoglobulin A nephropathy (IgAN) and persistent proteinuria remain at risk for kidney failure. We evaluated the efficacy and safety of hydroxychloroquine (HCQ), an immunomodulator, when added to the treatment regimen of patients with IgAN.
STUDY DESIGN: Double-blind, randomized, placebo-controlled, phase 2 clinical trial.
SETTING & PARTICIPANTS: Participants had IgAN (proteinuria with protein excretion of 0.75-3.5g/d and estimated glomerular filtration rate>30mL/min/1.73m2) and were receiving optimized RAAS inhibitor therapy.
INTERVENTIONS: Patients were randomly assigned 1:1 to receive daily oral HCQ or a placebo for 6 months.
OUTCOMES: The primary outcome was percentage change in proteinuria between baseline and 6 months.
RESULTS: 60 participants (mean estimated glomerular filtration rate, 53.8mL/min/1.73m2; median urine protein excretion, 1.7g/d) were recruited and randomly assigned to receive HCQ (n=30) or placebo (n=30). Percentage change in proteinuria at 6 months was significantly different between the HCQ group and the placebo group (-48.4% [IQR, -64.2%, -30.5%] vs 10.0% [IQR, -38.7%, 30.6%]; P<0.001, respectively). At 6 months, median proteinuria level was significantly lower in the HCQ group than in the placebo group (0.9 [IQR, 0.6, 1.0] g/d vs 1.9 [IQR, 0.9, 2.6] g/d; P=0.002, respectively). No serious adverse events were recorded during the study in either study group.
LIMITATIONS: The short treatment period and lack of postwithdrawal observations limit conclusions about long-term renoprotective efficacy and safety.
CONCLUSIONS: HCQ in addition to optimized RAAS inhibition significantly reduced proteinuria in patients with IgAN over 6 months without evidence of adverse events. These findings require confirmation in larger treatment trials.
FUNDING: This study was supported by grants from a government entity, the Capital of Clinical Characteristics, and the Applied Research Fund.
TRIAL REGISTRATION: Registered at ClinicalTrials.gov with study number NCT02942381.

...

### PMID 26510426 — pentoxifylline
**Effect of pentoxifylline in proteinuric chronic kidney disease: a systematic review and meta-analysis.**

Year: 2016  
Journal: Journal of nephrology  
Publication type: Journal Article; Meta-Analysis; Systematic Review  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/26510426/)

AIM: The aim of this study was to assess the effect of pentoxifylline on proteinuria and renal function in chronic kidney disease (CKD) treatment.
METHODS: We systematically searched PubMed, EMBASE, the Cochrane Library and ClinicalTrials.gov for randomized and non-randomized controlled trials comparing pentoxifylline to placebo, no treatment or renin-angiotensin system blockade in proteinuric CKD patients. The outcomes concerning proteinuria, renal function, blood pressure and adverse events were extracted.
RESULTS: Twelve trials with 613 participants were identified. Pentoxifylline significantly decreased proteinuria [weighted mean difference (WMD) -0.60 g/day (95 % CI -0.84 to -0.36); p < 0.001] compared to placebo or no-treatment groups, but the decrease was not significant [WMD: 0.10 g/day (-0.34 to 0.54); p = 0.66] compared to captopril treatment. The decrease of glomerular filtration rate was significantly less [WMD: 3.67 ml/min (2.71-4.62); p < 0.001] in the pentoxifylline group than in the controls. There was no significant difference in serum creatinine [WMD: -0.03 mg/dl (-0.10 to 0.03); p = 0.28], diastolic blood pressure [WMD: 0.94 mmHg (-0.74 to 2.61); p = 0.27] and adverse events [RR: 0.89 (0.60 to 1.32); p = 0.56].
CONCLUSIONS: Pentoxifylline may decrease proteinuria and protect renal function in patients with CKD. Further studies are needed to confirm this result.

### PMID 30949987 — pentoxifylline
**Pentoxifylline, progression of chronic kidney disease (CKD) and cardiovascular mortality: long-term follow-up of a randomized clinical trial.**

Year: 2019  
Journal: Journal of nephrology  
Publication type: Journal Article; Randomized Controlled Trial  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/30949987/)

BACKGROUND: Pentoxifylline could reduce proteinuria and slow renal disease progression. We previously conducted a single-blind, randomized, controlled trial that showed that pentoxifylline decreases inflammatory markers and stabilizes renal function.
SETTING AND PARTICIPANTS: 91 participants (46 in the pentoxifylline group and 45 in the control group) followed up for 7 additional years.
STUDY DESIGN: Post hoc analysis of a long-term follow-up after completion of the 12-months trial.
INTERVENTION: Pentoxifylline treatment (400 mg/twice a day) or standard treatment.
OUTCOME: Renal event (defined as starting dialysis therapy and/or doubling serum creatinine and/or ≥ 50% decrease in estimated glomerular filtration rate) and cardiovascular mortality.
RESULTS: During follow-up, a renal event was recorded in 24 patients from control group (13 initiated dialysis therapy and serum creatinine doubled in 11) and 11 patients from PTF group (7 initiated dialysis and serum creatinine doubled in 4) (log Rank: 5.822, p = 0.016). The possible protector effect of PTF was more significant in albuminuric patients and was independently of diabetes mellitus presence. Treatment with PTF reduced the renal events by 35% compared to the control group in a Cox model adjusted for diabetes mellitus, albuminuria and basal renal function (HR 0.65 (0.45-0.94), p = 0.022). Cardiovascular mortality was significantly reduced in PTF treatment (2 patients vs. 10 in control group) (log Rank 5.0977, p = 0.024). PTF treatment reduced cardiovascular mortality in 55% adjusted for diabetes mellitus and age (HR 0.45 (0.21-0.98), p = 0.044) (Table 3).
LIMITATIONS: Small sample size, single center, not double blind and post hoc follow-up analysis.
CONCLUSIONS: Long-term treatment with pentoxifylline may slow the rate of progression of kidney disease and reduce cardiovascular risk.

### PMID 33792832 — pentoxifylline
**A randomized double-blinded placebo-controlled clinical trial on protective effects of pentoxifylline on gentamicin nephrotoxicity in infectious patients.**

Year: 2021  
Journal: Clinical and experimental nephrology  
Publication type: Journal Article; Multicenter Study; Randomized Controlled Trial  
[Open PubMed record](https://pubmed.ncbi.nlm.nih.gov/33792832/)

BACKGROUND AND OBJECTIVE: Renal toxicity has limited gentamicin use in clinical practice. The aim of the present clinical trial was to assess the possible nephroprotective effects of pentoxifylline (PTX) against gentamicin nephrotoxicity.
MATERIALS AND METHODS: A multicenter, randomized, double-blind, placebo-controlled clinical trial was conducted on patients who had the indication for systemic gentamicin for at least 7 days. Sixty people were selected and randomly assigned. For patients in the intervention and control groups, 400 mg PTX sustained release tablet and placebo were given orally three times daily, respectively. Demographic, clinical, laboratory, and therapeutic information of patients were recorded. malondialdehyde (MDA) and tumor necrosis factor-alpha (TNF-α) levels in serum were measured on days 0 and 7.
RESULTS: The incidence of nephrotoxicity in the placebo group was 19.6 times higher than that in the PTX group (OR = 19.6, 95%CI = 3.08-114.32; P value = 0.001). The mean ± SD time onset of ATN was 4.00 ± 2.32 and 5.58 ± 1.59 days in PTX and placebo recipients, respectively (P value < 0.001). No significant differences were observed for hypokalemia, hypomagnesemia, potassium and magnesium wasting between the two groups. The mean ± SD levels of serum MDA and TNF-α at day 7 were significantly lower in the PTX compared to those in the placebo group (P value < 0.001 for both indexes).
CONCLUSION: The co-administration of 400 mg PTX orally three times daily along with gentamicin was both well-tolerated and effective in preventing the nephrotoxicity of gentamicin in patients with different infectious diseases.


Manual extraction worksheet saved: /kaggle/working/kidneyshield/results/manual_study_level_evidence_extraction.csv


,pmid,candidate_drug,title,year,journal,publication_types,pubmed_url,study_design_confirmed,drug_is_actual_intervention,population_and_disease,...,hard_kidney_outcome_reported,eGFR_slope_or_change_result,albuminuria_or_proteinuria_result,effect_estimate_and_95ci,statistical_significance,adverse_events_and_safety,major_bias_or_limitations,full_text_checked,reviewer_notes,evidence_category
0,21144773,pentoxifylline,Pentoxifylline for renoprotection in diabetic ...,2011,Journal of diabetes and its complications,Journal Article; Randomized Controlled Trial; ...,https://pubmed.ncbi.nlm.nih.gov/21144773/,,,,...,,,,,,,,,,
1,22336824,pentoxifylline,Pentoxifylline for diabetic kidney disease.,2012,The Cochrane database of systematic reviews,Journal Article; Meta-Analysis; Research Suppo...,https://pubmed.ncbi.nlm.nih.gov/22336824/,,,,...,,,,,,,,,,
2,35139160,colchicine,Colchicine use and the risk of CKD progression...,2022,"Rheumatology (Oxford, England)",Multicenter Study; Journal Article,https://pubmed.ncbi.nlm.nih.gov/35139160/,,,,...,,,,,,,,,,
3,30922594,hydroxychloroquine,Effects of Hydroxychloroquine on Proteinuria i...,2019,American journal of kidney diseases : the offi...,"Clinical Trial, Phase II; Journal Article; Ran...",https://pubmed.ncbi.nlm.nih.gov/30922594/,,,,...,,,,,,,,,,
4,35130544,allopurinol,The Effect of Allopurinol on Renal Outcomes in...,2022,Kidney & blood pressure research,Meta-Analysis; Systematic Review; Journal Article,https://pubmed.ncbi.nlm.nih.gov/35130544/,,,,...,,,,,,,,,,
5,29088122,allopurinol,Xanthine Oxidase Inhibitors for Improving Rena...,2017,International journal of molecular sciences,Journal Article; Meta-Analysis; Systematic Review,https://pubmed.ncbi.nlm.nih.gov/29088122/,,,,...,,,,,,,,,,
6,32579811,allopurinol,Effects of Allopurinol on the Progression of C...,2020,The New England journal of medicine,Journal Article; Multicenter Study; Randomized...,https://pubmed.ncbi.nlm.nih.gov/32579811/,,,,...,,,,,,,,,,
7,28794291,allopurinol,Allopurinol Against Progression of Chronic Kid...,2017,Iranian journal of kidney diseases,Journal Article; Randomized Controlled Trial,https://pubmed.ncbi.nlm.nih.gov/28794291/,,,,...,,,,,,,,,,
8,25928556,allopurinol,Urate lowering therapy to improve renal outcom...,2015,BMC nephrology,Journal Article; Meta-Analysis; Research Suppo...,https://pubmed.ncbi.nlm.nih.gov/25928556/,,,,...,,,,,,,,,,
9,16878429,pentoxifylline,Effects of pentoxifylline on the urinary prote...,2006,Clinical nephrology,Journal Article; Randomized Controlled Trial,https://pubmed.ncbi.nlm.nih.gov/16878429/,,,,...,,,,,,,,,,


In [8]:

# ============================================================
# KIDNEYSHIELD — CELL 6
# Curated, abstract-supported evidence matrix
# ============================================================

from pathlib import Path
import pandas as pd
from IPython.display import display

OUT = Path("/kaggle/working/kidneyshield/results")
OUT.mkdir(parents=True, exist_ok=True)

# Explicitly curated from the PubMed abstracts displayed in Cell 5.
# These are preliminary extractions, not full-text verified assessments.
evidence = [
    {
        "pmid": "21144773",
        "drug": "pentoxifylline",
        "study": "PREDIAN design and baseline report",
        "design": "Randomized controlled trial; design/baseline publication",
        "population": "169 adults with type 2 diabetes and CKD stages 3–4",
        "follow_up": "Planned 24 months",
        "endpoint": "Between-group eGFR at study end was the planned primary endpoint",
        "hard_kidney_outcome": "Not reported in this baseline paper",
        "surrogate_outcome": "Baseline albuminuria reported; no final treatment result in this paper",
        "abstract_result": "Intervention: 1200 mg/day added to RAS blockade; control n=87, intervention n=82",
        "evidence_category": "C — design paper, outcome results not reported here",
        "limitations": "Do not use this publication alone to estimate treatment efficacy",
        "next_action": "Link to the final PREDIAN results publication and extract its outcomes",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/21144773/",
    },
    {
        "pmid": "22336824",
        "drug": "pentoxifylline",
        "study": "Cochrane review of diabetic kidney disease",
        "design": "Systematic review of 17 studies, 991 participants",
        "population": "People with diabetic kidney disease",
        "follow_up": "Varied across included studies",
        "endpoint": "Kidney function and clinical outcomes",
        "hard_kidney_outcome": "No ESKD incidence or time-to-ESKD data available",
        "surrogate_outcome": "Some reductions in albuminuria/proteinuria; creatinine-clearance results not consistently significant",
        "abstract_result": "Review judged the evidence insufficient for recommendations",
        "evidence_category": "B — systematic review; clinical-outcome evidence insufficient",
        "limitations": "Included studies generally small, poorly reported, or at unclear/high risk of bias",
        "next_action": "Review newer evidence and assess overlap with later meta-analyses",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/22336824/",
    },
    {
        "pmid": "30949987",
        "drug": "pentoxifylline",
        "study": "Long-term follow-up of a randomized trial",
        "design": "Post-hoc long-term follow-up",
        "population": "91 participants; 46 treatment and 45 control",
        "follow_up": "Seven additional years after the original 12-month trial",
        "endpoint": "Renal events: dialysis, doubling of serum creatinine, or at least 50% eGFR decline",
        "hard_kidney_outcome": "Yes; composite renal events reported",
        "surrogate_outcome": "Prior trial reported inflammatory markers and renal-function measures",
        "abstract_result": "24 renal events in controls vs 11 in pentoxifylline group; adjusted HR 0.65 (95% CI 0.45–0.94)",
        "evidence_category": "A — direct outcome signal; certainty requires critical appraisal",
        "limitations": "Small single-center sample; single-blind original trial; post-hoc long-term analysis",
        "next_action": "Check original trial methods, event definitions, follow-up completeness, and adjusted model",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/30949987/",
    },
    {
        "pmid": "35139160",
        "drug": "colchicine",
        "study": "Colchicine use and CKD progression",
        "design": "Multicenter nested case-control observational study",
        "population": "CKD G3–G4 with hyperuricemia or chronic gout; 3,085 progression cases and 11,715 controls",
        "follow_up": "Varied; matched on follow-up time",
        "endpoint": "At least 40% eGFR decline or kidney failure requiring replacement therapy",
        "hard_kidney_outcome": "Yes; progression outcome included",
        "surrogate_outcome": "Not the main reported outcome",
        "abstract_result": "At least 90 cumulative daily doses associated with AOR 0.77 (95% CI 0.61–0.96) vs non-use",
        "evidence_category": "C — observational association",
        "limitations": "Residual confounding, treatment selection, and applicability limited to the studied population",
        "next_action": "Inspect exposure definitions, confounder adjustment, sensitivity analyses, and safety in CKD",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/35139160/",
    },
    {
        "pmid": "30922594",
        "drug": "hydroxychloroquine",
        "study": "Hydroxychloroquine in IgA nephropathy",
        "design": "Double-blind randomized placebo-controlled phase II trial",
        "population": "60 participants with IgA nephropathy and persistent proteinuria",
        "follow_up": "6 months",
        "endpoint": "Percentage change in proteinuria",
        "hard_kidney_outcome": "No long-term kidney-failure benefit established in this trial",
        "surrogate_outcome": "Yes; proteinuria",
        "abstract_result": "Proteinuria change −48.4% vs +10.0%; P<0.001",
        "evidence_category": "B — randomized surrogate-outcome evidence",
        "limitations": "Small sample, short treatment period, and no post-withdrawal observations",
        "next_action": "Verify full-text safety data and whether subsequent confirmatory trials exist",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/30922594/",
    },
    {
        "pmid": "32579811",
        "drug": "allopurinol",
        "study": "CKD-FIX",
        "design": "Multicenter randomized controlled trial",
        "population": "363 assessed participants with stage 3–4 CKD at high risk of progression and no gout history",
        "follow_up": "104 weeks",
        "endpoint": "Change in eGFR",
        "hard_kidney_outcome": "Primary endpoint was eGFR change, not kidney failure alone",
        "surrogate_outcome": "eGFR slope/change",
        "abstract_result": "eGFR decline −3.33 vs −3.23 mL/min/1.73 m²/year; difference −0.10 (95% CI −1.18 to 0.97); P=0.85",
        "evidence_category": "A — direct randomized test of kidney-function progression; null result",
        "limitations": "Enrollment stopped early; results apply to the trial population and intervention",
        "next_action": "Retain as important evidence against assuming urate lowering slows CKD",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/32579811/",
    },
    {
        "pmid": "35130544",
        "drug": "allopurinol",
        "study": "Meta-analysis in diabetic kidney disease",
        "design": "Systematic review and meta-analysis of 3 trials",
        "population": "Patients with diabetic kidney disease",
        "follow_up": "Varied across trials",
        "endpoint": "Change in GFR; albuminuria and uric acid secondary outcomes",
        "hard_kidney_outcome": "No hard-outcome benefit established by the abstract",
        "surrogate_outcome": "GFR and albuminuria",
        "abstract_result": "No significant GFR improvement: WMD 1.07 (95% CI −1.68 to 3.82); P=0.45",
        "evidence_category": "B — synthesis of kidney-function and surrogate outcomes",
        "limitations": "Only three trials included; findings should be interpreted alongside larger individual trials",
        "next_action": "Check included trials and overlap with CKD-FIX and other reviews",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/35130544/",
    },
    {
        "pmid": "29088122",
        "drug": "allopurinol",
        "study": "Xanthine oxidase inhibitors meta-analysis",
        "design": "Systematic review and meta-analysis",
        "population": "CKD populations in randomized trials of xanthine oxidase inhibitors",
        "follow_up": "Varied across included studies",
        "endpoint": "ESKD, eGFR, creatinine, proteinuria, and albuminuria",
        "hard_kidney_outcome": "Review reported a pooled ESKD signal in three small studies",
        "surrogate_outcome": "Pooled eGFR signal in some trial subsets",
        "abstract_result": "ESKD RR 0.42 (95% CI 0.22–0.80) in 204 participants; review called for confirmatory trials",
        "evidence_category": "B — older pooled evidence, requiring reconciliation with later trials",
        "limitations": "Small pooled sample and limited generalizability; xanthine oxidase inhibitor class is not synonymous with allopurinol alone",
        "next_action": "Inspect trial-level data and compare with later, larger negative trials",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/29088122/",
    },
    {
        "pmid": "28415944",
        "drug": "pentoxifylline",
        "study": "Pentoxifylline plus ACEI/ARB meta-analysis",
        "design": "Meta-analysis of 11 RCTs, 705 participants",
        "population": "CKD patients receiving ACE inhibitors or ARBs",
        "follow_up": "Several durations; subgroup results reported at 6 and 9–12 months",
        "endpoint": "Proteinuria and kidney-function measures",
        "hard_kidney_outcome": "No hard kidney-failure benefit established in the abstract",
        "surrogate_outcome": "Proteinuria and eGFR",
        "abstract_result": "Reported favorable proteinuria/eGFR findings; some confidence-limit reporting in the abstract warrants source verification",
        "evidence_category": "B — pooled surrogate/kidney-function evidence",
        "limitations": "Heterogeneity and short follow-up; verify effect direction, confidence intervals, and units in the full text",
        "next_action": "Extract trial-level eGFR outcomes and check all reported confidence intervals",
        "source_url": "https://pubmed.ncbi.nlm.nih.gov/28415944/",
    },
]

df = pd.DataFrame(evidence)

# Add explicit review-state fields to avoid confusing abstract extraction with
# completed full-text appraisal.
df["extraction_status"] = "Preliminary — PubMed abstract"
df["full_text_verified"] = "No"
df["risk_of_bias_assessment_complete"] = "No"
df["clinical_recommendation"] = "Not assessed"

csv_path = OUT / "kidneyshield_curated_evidence_matrix.csv"
df.to_csv(csv_path, index=False)

print("Evidence matrix saved:", csv_path)
print("Studies:", len(df))
print("\nDrug-level evidence inventory:")
display(
    df.groupby(["drug", "evidence_category"], dropna=False)
      .size()
      .rename("studies")
      .reset_index()
)

print("\nMain comparison:")
display(df[[
    "drug", "pmid", "study", "design",
    "hard_kidney_outcome", "surrogate_outcome",
    "abstract_result", "evidence_category",
    "limitations", "source_url"
]])

print("\nImportant: this is a preliminary evidence map, not a clinical recommendation.")

Evidence matrix saved: /kaggle/working/kidneyshield/results/kidneyshield_curated_evidence_matrix.csv
Studies: 9

Drug-level evidence inventory:


,drug,evidence_category,studies
0,allopurinol,A — direct randomized test of kidney-function ...,1
1,allopurinol,"B — older pooled evidence, requiring reconcili...",1
2,allopurinol,B — synthesis of kidney-function and surrogate...,1
3,colchicine,C — observational association,1
4,hydroxychloroquine,B — randomized surrogate-outcome evidence,1
5,pentoxifylline,A — direct outcome signal; certainty requires ...,1
6,pentoxifylline,B — pooled surrogate/kidney-function evidence,1
7,pentoxifylline,B — systematic review; clinical-outcome eviden...,1
8,pentoxifylline,"C — design paper, outcome results not reported...",1



Main comparison:


,drug,pmid,study,design,hard_kidney_outcome,surrogate_outcome,abstract_result,evidence_category,limitations,source_url
0,pentoxifylline,21144773,PREDIAN design and baseline report,Randomized controlled trial; design/baseline p...,Not reported in this baseline paper,Baseline albuminuria reported; no final treatm...,Intervention: 1200 mg/day added to RAS blockad...,"C — design paper, outcome results not reported...",Do not use this publication alone to estimate ...,https://pubmed.ncbi.nlm.nih.gov/21144773/
1,pentoxifylline,22336824,Cochrane review of diabetic kidney disease,"Systematic review of 17 studies, 991 participants",No ESKD incidence or time-to-ESKD data available,Some reductions in albuminuria/proteinuria; cr...,Review judged the evidence insufficient for re...,B — systematic review; clinical-outcome eviden...,"Included studies generally small, poorly repor...",https://pubmed.ncbi.nlm.nih.gov/22336824/
2,pentoxifylline,30949987,Long-term follow-up of a randomized trial,Post-hoc long-term follow-up,Yes; composite renal events reported,Prior trial reported inflammatory markers and ...,24 renal events in controls vs 11 in pentoxify...,A — direct outcome signal; certainty requires ...,Small single-center sample; single-blind origi...,https://pubmed.ncbi.nlm.nih.gov/30949987/
3,colchicine,35139160,Colchicine use and CKD progression,Multicenter nested case-control observational ...,Yes; progression outcome included,Not the main reported outcome,At least 90 cumulative daily doses associated ...,C — observational association,"Residual confounding, treatment selection, and...",https://pubmed.ncbi.nlm.nih.gov/35139160/
4,hydroxychloroquine,30922594,Hydroxychloroquine in IgA nephropathy,Double-blind randomized placebo-controlled pha...,No long-term kidney-failure benefit establishe...,Yes; proteinuria,Proteinuria change −48.4% vs +10.0%; P<0.001,B — randomized surrogate-outcome evidence,"Small sample, short treatment period, and no p...",https://pubmed.ncbi.nlm.nih.gov/30922594/
5,allopurinol,32579811,CKD-FIX,Multicenter randomized controlled trial,"Primary endpoint was eGFR change, not kidney f...",eGFR slope/change,eGFR decline −3.33 vs −3.23 mL/min/1.73 m²/yea...,A — direct randomized test of kidney-function ...,Enrollment stopped early; results apply to the...,https://pubmed.ncbi.nlm.nih.gov/32579811/
6,allopurinol,35130544,Meta-analysis in diabetic kidney disease,Systematic review and meta-analysis of 3 trials,No hard-outcome benefit established by the abs...,GFR and albuminuria,No significant GFR improvement: WMD 1.07 (95% ...,B — synthesis of kidney-function and surrogate...,Only three trials included; findings should be...,https://pubmed.ncbi.nlm.nih.gov/35130544/
7,allopurinol,29088122,Xanthine oxidase inhibitors meta-analysis,Systematic review and meta-analysis,Review reported a pooled ESKD signal in three ...,Pooled eGFR signal in some trial subsets,ESKD RR 0.42 (95% CI 0.22–0.80) in 204 partici...,"B — older pooled evidence, requiring reconcili...",Small pooled sample and limited generalizabili...,https://pubmed.ncbi.nlm.nih.gov/29088122/
8,pentoxifylline,28415944,Pentoxifylline plus ACEI/ARB meta-analysis,"Meta-analysis of 11 RCTs, 705 participants",No hard kidney-failure benefit established in ...,Proteinuria and eGFR,Reported favorable proteinuria/eGFR findings; ...,B — pooled surrogate/kidney-function evidence,Heterogeneity and short follow-up; verify effe...,https://pubmed.ncbi.nlm.nih.gov/28415944/



Important: this is a preliminary evidence map, not a clinical recommendation.


In [9]:

from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/kidneyshield")
RESULTS = BASE / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

matrix_path = RESULTS / "kidneyshield_curated_evidence_matrix.csv"
df = pd.read_csv(matrix_path)

# Add the final PREDIAN outcomes publication.
predian = {
    "drug": "pentoxifylline",
    "pmid": 24970885,
    "study": "PREDIAN final randomized-trial results",
    "design": (
        "Single-center, open-label, prospective randomized controlled trial; "
        "169 participants with type 2 diabetes and stage 3–4 CKD; "
        "82 assigned pentoxifylline and 87 control"
    ),
    "hard_kidney_outcome": (
        "No kidney-failure endpoint established by this study; "
        "primary outcome was change in eGFR"
    ),
    "surrogate_outcome": (
        "eGFR change and urinary albumin excretion"
    ),
    "abstract_result": (
        "Over approximately 2 years, mean eGFR decline was 2.1 ± 0.4 "
        "mL/min/1.73 m² with pentoxifylline versus 6.5 ± 0.4 in controls; "
        "between-group difference 4.3 (95% CI 3.1–5.5), P<0.001. "
        "Urinary albumin excretion also favored pentoxifylline."
    ),
    "evidence_category": (
        "A — randomized kidney-function outcome; "
        "certainty and generalizability require appraisal"
    ),
    "limitations": (
        "Single-center, open-label trial; selected population with "
        "type 2 diabetes and stage 3–4 CKD; eGFR outcome rather than "
        "a demonstrated reduction in kidney failure; independent "
        "replication and full risk-of-bias assessment needed"
    ),
    "source_url": "https://pubmed.ncbi.nlm.nih.gov/24970885/",
    "extraction_status": "Preliminary — PubMed abstract",
    "full_text_verified": "No",
    "risk_of_bias": "Not fully assessed",
    "clinical_recommendation": "None"
}

# Align the new record to the existing matrix schema.
for col in df.columns:
    if col not in predian:
        predian[col] = None

# Avoid duplicate insertion if this cell is run again.
df = df[df["pmid"].astype(str) != str(predian["pmid"])]
df = pd.concat([df, pd.DataFrame([predian])[df.columns]], ignore_index=True)

df["pmid"] = df["pmid"].astype(str)
df = df.drop_duplicates(subset=["pmid", "drug"], keep="last")
df = df.sort_values(["drug", "pmid"]).reset_index(drop=True)

df.to_csv(matrix_path, index=False)

print("Updated evidence matrix:", matrix_path)
print("Total drug-study records:", len(df))
print("\nPentoxifylline records:")
display(
    df.loc[df["drug"].eq("pentoxifylline"),
           ["pmid", "study", "design", "abstract_result",
            "evidence_category", "full_text_verified"]]
)

Updated evidence matrix: /kaggle/working/kidneyshield/results/kidneyshield_curated_evidence_matrix.csv
Total drug-study records: 10

Pentoxifylline records:


,pmid,study,design,abstract_result,evidence_category,full_text_verified
5,21144773,PREDIAN design and baseline report,Randomized controlled trial; design/baseline p...,Intervention: 1200 mg/day added to RAS blockad...,"C — design paper, outcome results not reported...",No
6,22336824,Cochrane review of diabetic kidney disease,"Systematic review of 17 studies, 991 participants",Review judged the evidence insufficient for re...,B — systematic review; clinical-outcome eviden...,No
7,24970885,PREDIAN final randomized-trial results,"Single-center, open-label, prospective randomi...","Over approximately 2 years, mean eGFR decline ...",A — randomized kidney-function outcome; certai...,No
8,28415944,Pentoxifylline plus ACEI/ARB meta-analysis,"Meta-analysis of 11 RCTs, 705 participants",Reported favorable proteinuria/eGFR findings; ...,B — pooled surrogate/kidney-function evidence,No
9,30949987,Long-term follow-up of a randomized trial,Post-hoc long-term follow-up,24 renal events in controls vs 11 in pentoxify...,A — direct outcome signal; certainty requires ...,No


In [11]:

from pathlib import Path
import pandas as pd

BASE = Path("/kaggle/working/kidneyshield")
RESULTS = BASE / "results"

matrix_path = RESULTS / "kidneyshield_curated_evidence_matrix.csv"
output_path = RESULTS / "kidneyshield_critical_appraisal.csv"

df = pd.read_csv(matrix_path)
df["pmid"] = df["pmid"].astype(str).str.replace(r"\.0$", "", regex=True)
df["full_text_verified"] = df["full_text_verified"].fillna("No")

# Fields requiring structured appraisal rather than keyword scoring.
appraisal_fields = {
    "population_details_verified": "",
    "sample_size_verified": "",
    "random_sequence_generation": "Not assessed",
    "allocation_concealment": "Not assessed",
    "blinding": "Not assessed",
    "comparator_appropriate": "Not assessed",
    "follow_up_adequate": "Not assessed",
    "attrition_handled_appropriately": "Not assessed",
    "prespecified_primary_outcome_verified": "Not assessed",
    "hard_kidney_outcome_reported": "Not assessed",
    "eGFR_outcome_reported": "Not assessed",
    "albuminuria_or_proteinuria_outcome_reported": "Not assessed",
    "effect_estimate_verified": "Not assessed",
    "confidence_interval_verified": "Not assessed",
    "risk_of_bias_overall": "Not assessed",
    "appraisal_notes": "",
    "review_status": "Awaiting full-text/primary-source review",
}

for col, default in appraisal_fields.items():
    if col not in df.columns:
        df[col] = default

# Keep the original evidence fields and add appraisal columns.
df.to_csv(output_path, index=False)

print("Critical-appraisal worksheet saved:", output_path)
print("Records:", len(df))
print("\nReview status:")
display(df["review_status"].value_counts().rename_axis("status").to_frame("records"))

print("\nPrioritize these records for full-text verification:")
priority_pmids = [
    "24970885",  # PREDIAN final results
    "30949987",  # long-term follow-up
    "32579811",  # CKD-FIX
    "35139160",  # colchicine observational study
    "30922594",  # hydroxychloroquine RCT
    "22336824",  # Cochrane review
]
display(
    df[df["pmid"].isin(priority_pmids)][
        ["drug", "pmid", "study", "design", "full_text_verified",
         "risk_of_bias_overall", "review_status"]
    ].sort_values(["drug", "pmid"])
)

Critical-appraisal worksheet saved: /kaggle/working/kidneyshield/results/kidneyshield_critical_appraisal.csv
Records: 10

Review status:


,records
status,
Awaiting full-text/primary-source review,10



Prioritize these records for full-text verification:


,drug,pmid,study,design,full_text_verified,risk_of_bias_overall,review_status
1,allopurinol,32579811,CKD-FIX,Multicenter randomized controlled trial,No,Not assessed,Awaiting full-text/primary-source review
3,colchicine,35139160,Colchicine use and CKD progression,Multicenter nested case-control observational ...,No,Not assessed,Awaiting full-text/primary-source review
4,hydroxychloroquine,30922594,Hydroxychloroquine in IgA nephropathy,Double-blind randomized placebo-controlled pha...,No,Not assessed,Awaiting full-text/primary-source review
6,pentoxifylline,22336824,Cochrane review of diabetic kidney disease,"Systematic review of 17 studies, 991 participants",No,Not assessed,Awaiting full-text/primary-source review
7,pentoxifylline,24970885,PREDIAN final randomized-trial results,"Single-center, open-label, prospective randomi...",No,Not assessed,Awaiting full-text/primary-source review
9,pentoxifylline,30949987,Long-term follow-up of a randomized trial,Post-hoc long-term follow-up,No,Not assessed,Awaiting full-text/primary-source review


In [13]:

from pathlib import Path
import pandas as pd

path = Path("/kaggle/working/kidneyshield/results/kidneyshield_critical_appraisal.csv")
df = pd.read_csv(path, dtype={"pmid": str})
df["pmid"] = df["pmid"].str.replace(r"\.0$", "", regex=True)

records = {
    "24970885": {
        "source_record_checked": "Yes — PubMed abstract",
        "full_text_available_url": "https://pmc.ncbi.nlm.nih.gov/articles/PMC4279740/",
        "sample_size_verified": "169 randomized; 82 pentoxifylline, 87 control",
        "effect_estimate_verified": "Abstract: eGFR decline 2.1 vs 6.5 mL/min/1.73 m²; difference 4.3",
        "confidence_interval_verified": "Abstract reports 95% CI 3.1 to 5.5",
        "risk_of_bias_overall": "Provisional: concerns due to open-label, single-center design; full appraisal pending",
        "related_publication_group": "PREDIAN trial",
        "independent_randomized_trial": "Yes — original randomized trial",
        "appraisal_notes": "Review full text for randomization, allocation concealment, missing data, analysis population, outcome measurement, and adverse events.",
    },
    "30949987": {
        "source_record_checked": "Yes — PubMed abstract",
        "full_text_available_url": "https://pubmed.ncbi.nlm.nih.gov/30949987/",
        "sample_size_verified": "91 participants in long-term follow-up",
        "effect_estimate_verified": "Abstract: adjusted HR 0.65 for renal events",
        "confidence_interval_verified": "Abstract reports 95% CI 0.45 to 0.94",
        "risk_of_bias_overall": "Important concerns: small cohort and post-hoc long-term follow-up; full appraisal pending",
        "related_publication_group": "PREDIAN-related pentoxifylline cohort",
        "independent_randomized_trial": "No — follow-up of an earlier trial cohort",
        "hard_kidney_outcome_reported": "Composite renal events reported; verify exact definition and adjudication in full text",
        "appraisal_notes": "Confirm follow-up completeness, event definitions, adjusted model, treatment exposure, and relationship to the original PREDIAN trial.",
    },
    "32579811": {
        "source_record_checked": "Yes — PubMed abstract and NEJM article record",
        "full_text_available_url": "https://www.nejm.org/doi/full/10.1056/NEJMoa1915833",
        "sample_size_verified": "369 randomized; 363 included in primary outcome assessment",
        "effect_estimate_verified": "Mean difference in eGFR change: −0.10 mL/min/1.73 m²/year",
        "confidence_interval_verified": "95% CI −1.18 to 0.97; P=0.85",
        "risk_of_bias_overall": "Formal assessment pending; recruitment ended early because of slow enrollment",
        "related_publication_group": "CKD-FIX",
        "independent_randomized_trial": "Yes — original randomized trial",
        "appraisal_notes": "Review missing data, adherence, outcome analysis, and precision in light of early recruitment termination.",
    },
}

for field in set().union(*(r.keys() for r in records.values())):
    if field not in df.columns:
        df[field] = ""
    df[field] = df[field].astype("string")

for pmid, values in records.items():
    mask = df["pmid"].eq(pmid)
    if not mask.any():
        print("WARNING: PMID not found:", pmid)
        continue
    for field, value in values.items():
        df.loc[mask, field] = value

# Keep source verification distinct from completed full-text review.
if "full_text_verified" in df.columns:
    df["full_text_verified"] = df["full_text_verified"].fillna("No")
    df.loc[df["pmid"].isin(records), "full_text_verified"] = (
        "No — full text still requires appraisal"
    )

df.to_csv(path, index=False)

check = pd.read_csv(path, dtype={"pmid": str})
check["pmid"] = check["pmid"].str.replace(r"\.0$", "", regex=True)

cols = [
    "pmid", "study", "sample_size_verified",
    "effect_estimate_verified", "confidence_interval_verified",
    "risk_of_bias_overall", "related_publication_group",
    "independent_randomized_trial", "full_text_verified"
]
cols = [c for c in cols if c in check.columns]

display(check[check["pmid"].isin(records)][cols].sort_values("pmid"))
print("\nSaved and reloaded:", path)

,pmid,study,sample_size_verified,effect_estimate_verified,confidence_interval_verified,risk_of_bias_overall,related_publication_group,independent_randomized_trial,full_text_verified
7,24970885,PREDIAN final randomized-trial results,"169 randomized; 82 pentoxifylline, 87 control",Abstract: eGFR decline 2.1 vs 6.5 mL/min/1.73 ...,Abstract reports 95% CI 3.1 to 5.5,"Provisional: concerns due to open-label, singl...",PREDIAN trial,Yes — original randomized trial,No — full text still requires appraisal
9,30949987,Long-term follow-up of a randomized trial,91 participants in long-term follow-up,Abstract: adjusted HR 0.65 for renal events,Abstract reports 95% CI 0.45 to 0.94,Important concerns: small cohort and post-hoc ...,PREDIAN-related pentoxifylline cohort,No — follow-up of an earlier trial cohort,No — full text still requires appraisal
1,32579811,CKD-FIX,369 randomized; 363 included in primary outcom...,Mean difference in eGFR change: −0.10 mL/min/1...,95% CI −1.18 to 0.97; P=0.85,Formal assessment pending; recruitment ended e...,CKD-FIX,Yes — original randomized trial,No — full text still requires appraisal



Saved and reloaded: /kaggle/working/kidneyshield/results/kidneyshield_critical_appraisal.csv


In [14]:

from pathlib import Path
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE = Path("/kaggle/working/kidneyshield")
RAW = BASE / "raw_data"
RESULTS = BASE / "results"
RAW.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

url = "https://pmc.ncbi.nlm.nih.gov/articles/PMC4279740/"
out_txt = RAW / "predian_fulltext_extracted.txt"

try:
    response = requests.get(
        url,
        timeout=30,
        headers={"User-Agent": "KidneyShieldEvidenceResearch/1.0"}
    )
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("content-type", "unknown"))

    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Remove navigation and page furniture before extracting visible text.
    for tag in soup(["script", "style", "nav", "header", "footer"]):
        tag.decompose()

    article = soup.find("article") or soup.find("main") or soup.body
    text = article.get_text("\n", strip=True) if article else ""

    if len(text) > 5000:
        out_txt.write_text(text, encoding="utf-8")
        print("Extracted text saved:", out_txt)
        print("Character count:", len(text))
        print("\nPreview:\n")
        print(text[:6000])
    else:
        print("Response did not yield enough article text.")
        print("Preview:\n", text[:1500])

except Exception as e:
    print("Retrieval failed:", repr(e))
    print(
        "Open the article manually if internet access is unavailable: "
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC4279740/"
    )

print(
    "\nDo not set full_text_verified='Yes' solely because the page downloaded. "
    "Verify the relevant methods and results sections before updating the appraisal."
)

HTTP status: 200
Content type: text/html; charset=utf-8
Extracted text saved: /kaggle/working/kidneyshield/raw_data/predian_fulltext_extracted.txt
Character count: 52717

Preview:

J Am Soc Nephrol
. 2014 Jun 26;26(1):220–229. doi:
10.1681/ASN.2014010012
Effect of Pentoxifylline on Renal Function and Urinary Albumin Excretion in Patients with Diabetic Kidney Disease: The PREDIAN Trial
Juan F Navarro-González
Juan F Navarro-González
*
Nephrology Service,
†
Research Unit,
‡
GEENDIAB (Spanish Group for the Study of Diabetic Nephropathy), University Hospital Nuestra Señora de Candelaria, Santa Cruz de Tenerife, Spain
Find articles by
Juan F Navarro-González
*,
†,
‡,
✉
,
Carmen Mora-Fernández
Carmen Mora-Fernández
†
Research Unit,
‡
GEENDIAB (Spanish Group for the Study of Diabetic Nephropathy), University Hospital Nuestra Señora de Candelaria, Santa Cruz de Tenerife, Spain
Find articles by
Carmen Mora-Fernández
†,
‡
,
Mercedes Muros de Fuentes
Mercedes Muros de Fuentes
‡
GEENDIAB (Spanish 

In [15]:

from pathlib import Path
import re

BASE = Path("/kaggle/working/kidneyshield")
source = BASE / "raw_data" / "predian_fulltext_extracted.txt"
out = BASE / "results" / "predian_review_excerpts.txt"
out.parent.mkdir(parents=True, exist_ok=True)

if not source.exists():
    raise FileNotFoundError(f"Full-text file not found: {source}")

text = source.read_text(encoding="utf-8", errors="replace")
text = re.sub(r"\n{3,}", "\n\n", text)

# Find likely section headings and relevant methods/safety phrases.
heading_patterns = [
    r"Materials and Methods",
    r"Methods",
    r"Study Design",
    r"Study Population",
    r"Randomization",
    r"Statistical Analyses",
    r"Results",
    r"Adverse Events",
    r"Discussion",
    r"Limitations",
]

print("ARTICLE SECTION HEADING MATCHES")
for pattern in heading_patterns:
    matches = list(re.finditer(pattern, text, flags=re.I))
    print(f"\n{pattern}: {len(matches)} match(es)")
    for m in matches[:3]:
        print("  ", repr(text[max(0, m.start()-70):m.end()+140].replace("\n", " ")))

# Extract windows around the most important appraisal terms.
search_terms = [
    "randomized", "allocation", "blinding", "open-label",
    "intention-to-treat", "withdraw", "lost to follow-up",
    "missing data", "adverse event", "side effect",
    "limitation", "primary outcome", "sample size",
    "power calculation", "eGFR", "creatinine"
]

chunks = []
for term in search_terms:
    matches = list(re.finditer(re.escape(term), text, flags=re.I))
    if matches:
        chunks.append(f"\n{'='*24}\nTERM: {term} | matches: {len(matches)}\n{'='*24}")
        # Print at most two contexts per term to keep output manageable.
        for m in matches[:2]:
            start = max(0, m.start() - 450)
            end = min(len(text), m.end() + 950)
            chunks.append(text[start:end].strip())

result = "\n\n".join(chunks)
out.write_text(result, encoding="utf-8")

print("\n\nTargeted excerpts saved to:", out)
print("\nTARGETED EXCERPTS\n")
print(result[:24000])

ARTICLE SECTION HEADING MATCHES

Materials and Methods: 0 match(es)

Methods: 2 match(es)
   'e was assessed by measuring eGFR; therefore, the use of more accurate methods for determination of the GFR would be important. In addition, the use of eGFR as an endpoint may be a potential limitation because ideally,'
   'ence the renoprotective properties of PTF in this population. Concise Methods Study Design The PREDIAN (Pentoxifylline for Renoprotection in Diabetic Nephropathy) study 42 is an independent, investigator-initiated tri'

Study Design: 1 match(es)
   ' renoprotective properties of PTF in this population. Concise Methods Study Design The PREDIAN (Pentoxifylline for Renoprotection in Diabetic Nephropathy) study 42 is an independent, investigator-initiated trial without an'

Study Population: 0 match(es)

Randomization: 5 match(es)
   '. Open in a new tab Evolution of average systolic and diastolic BP at randomization (basal) and during follow-up in the control and PTF groups wit

In [16]:

from pathlib import Path
import pandas as pd

path = Path("/kaggle/working/kidneyshield/results/kidneyshield_critical_appraisal.csv")

df = pd.read_csv(path, dtype={"pmid": "string"})
df["pmid"] = df["pmid"].str.replace(r"\.0$", "", regex=True)

updates = {
    "source_record_checked":
        "Full-text article retrieved; Methods, Results, safety, and limitations excerpts reviewed",
    "full_text_verified":
        "Retrieved; key sections reviewed; complete appraisal pending",
    "population_details_verified":
        "Adults with type 2 diabetes and stage 3–4 CKD; mean baseline eGFR 37.4 mL/min/1.73 m²; all participants reported as white",
    "sample_size_verified":
        "169 randomized: 82 pentoxifylline, 87 control; 462 initially screened",
    "random_sequence_generation":
        "Computer-generated random numbers; described as simple randomization",
    "allocation_concealment":
        "Sequentially numbered, opaque, sealed envelopes; opened after baseline assessments",
    "blinding":
        "Open-label; no placebo control. Laboratory measurements for main outcomes were blinded to group allocation",
    "comparator_appropriate":
        "Control group continued RAS inhibitor treatment; authors report similar RAS inhibitor doses between groups",
    "follow_up_adequate":
        "Planned and reported treatment period: 2 years; participant-flow details still need review",
    "attrition_handled_appropriately":
        "Not fully assessed; inspect participant flow and missing outcome data",
    "prespecified_primary_outcome_verified":
        "Authors describe eGFR change as the primary outcome; protocol/registration verification pending",
    "hard_kidney_outcome_reported":
        "Kidney failure was not the primary outcome; study was not definitive for kidney-failure prevention",
    "eGFR_outcome_reported":
        "Yes; mean eGFR decline 2.1 vs 6.5 mL/min/1.73 m² over 2 years; between-group difference 4.3 (95% CI 3.1–5.5), P<0.001",
    "albuminuria_or_proteinuria_outcome_reported":
        "Yes; urinary albumin excretion percentage change −14.9% with pentoxifylline vs +5.7% control; P=0.001",
    "effect_estimate_verified":
        "Full-text abstract/results excerpt reviewed; eGFR and urinary albumin results recorded",
    "confidence_interval_verified":
        "eGFR difference 95% CI 3.1–5.5 mL/min/1.73 m²; albumin-excretion group-specific CIs also reported",
    "risk_of_bias_overall":
        "Provisional: some concerns. Strengths include computer-generated randomization, concealed allocation, and blinded laboratory measurements. Concerns include open-label/no placebo, single-center design, eGFR primary endpoint, and attrition/missing-data details not yet fully checked.",
    "appraisal_notes":
        "Safety: gastrointestinal symptoms 21.9% vs 10.3% (P=0.03); one participant discontinued pentoxifylline because of symptoms, and five could not increase to 1200 mg/day because of digestive intolerance. Authors acknowledge open-label, no-placebo, single-center, and eGFR-endpoint limitations. Verify participant flow, analysis population, missing data, and protocol before final rating.",
    "review_status":
        "Partial full-text review complete; participant flow and missing-data appraisal pending",
}

mask = df["pmid"].eq("24970885")
if not mask.any():
    raise ValueError("PMID 24970885 not found in appraisal worksheet.")

# Cast every updated column to a compatible text dtype before assignment.
for col, value in updates.items():
    if col not in df.columns:
        df[col] = pd.Series("", index=df.index, dtype="string")
    else:
        df[col] = df[col].astype("string")
    df.loc[mask, col] = value

df.to_csv(path, index=False)

# Reload to confirm the saved record.
check = pd.read_csv(path, dtype={"pmid": "string"})
check["pmid"] = check["pmid"].str.replace(r"\.0$", "", regex=True)
row = check.loc[check["pmid"].eq("24970885")]

print("Saved:", path)
display(row[[
    "pmid", "study", "random_sequence_generation",
    "allocation_concealment", "blinding",
    "eGFR_outcome_reported", "risk_of_bias_overall",
    "review_status"
]])

Saved: /kaggle/working/kidneyshield/results/kidneyshield_critical_appraisal.csv


,pmid,study,random_sequence_generation,allocation_concealment,blinding,eGFR_outcome_reported,risk_of_bias_overall,review_status
7,24970885,PREDIAN final randomized-trial results,Computer-generated random numbers; described a...,"Sequentially numbered, opaque, sealed envelope...",Open-label; no placebo control. Laboratory mea...,Yes; mean eGFR decline 2.1 vs 6.5 mL/min/1.73 ...,Provisional: some concerns. Strengths include ...,Partial full-text review complete; participant...


In [17]:

# NEXT CELL — KidneyShield: PREDIAN evidence verification queue
# Read-only for the existing appraisal worksheet.
# Creates a separate template for recording source-verified findings.

from pathlib import Path
import pandas as pd
import json

ROOT = Path("/kaggle/working/kidneyshield")
RESULTS = ROOT / "results"
APPRAISAL = RESULTS / "kidneyshield_critical_appraisal.csv"

assert APPRAISAL.exists(), f"Missing file: {APPRAISAL}"

df = pd.read_csv(APPRAISAL, dtype={"pmid": "string"})
df["pmid"] = (
    df["pmid"].str.strip().str.replace(r"\.0$", "", regex=True)
)

pmid = "24970885"
study_rows = df.loc[df["pmid"].eq(pmid)]

if len(study_rows) != 1:
    raise ValueError(
        f"Expected exactly one record for PMID {pmid}; "
        f"found {len(study_rows)}."
    )

study = study_rows.iloc[0]

# Source-verification tasks: fill only after checking the cited source.
verification_tasks = [
    {
        "priority": 1,
        "domain": "Participant flow",
        "question": "How many randomized participants completed follow-up in each arm?",
        "required_evidence": "CONSORT diagram or participant-flow section",
    },
    {
        "priority": 1,
        "domain": "Analysis population",
        "question": "Was the primary analysis intention-to-treat, per-protocol, or another population?",
        "required_evidence": "Methods and statistical-analysis section",
    },
    {
        "priority": 1,
        "domain": "Missing data",
        "question": "How many primary-outcome measurements were missing, and how were they handled?",
        "required_evidence": "Methods, Results, tables, and supplementary material",
    },
    {
        "priority": 1,
        "domain": "Protocol and registration",
        "question": "Can the trial registration or protocol confirm the prespecified primary outcome and analysis?",
        "required_evidence": "Trial registry entry and protocol, if available",
    },
    {
        "priority": 2,
        "domain": "Outcome estimates",
        "question": "Are the eGFR estimate, confidence interval, units, and analysis method transcribed correctly?",
        "required_evidence": "Original Results text and tables",
    },
    {
        "priority": 2,
        "domain": "Albuminuria",
        "question": "What were the exact group estimates, confidence intervals, denominators, and statistical test?",
        "required_evidence": "Original Results text and tables",
    },
    {
        "priority": 2,
        "domain": "Safety",
        "question": "Were serious adverse events and adverse-event discontinuations reported for both arms?",
        "required_evidence": "Safety section and supplementary tables",
    },
    {
        "priority": 2,
        "domain": "Applicability",
        "question": "How closely does this study population match the intended CKD population for KidneyShield?",
        "required_evidence": "Eligibility criteria, baseline characteristics, and limitations",
    },
]

template = pd.DataFrame(verification_tasks)
template["source_location"] = ""
template["verified_finding"] = ""
template["evidence_quote_or_table"] = ""
template["reviewer_status"] = "Not yet verified"
template["review_notes"] = ""

template_path = RESULTS / "predian_source_verification_template.csv"
template.to_csv(template_path, index=False)

# Snapshot current worksheet status without altering the original.
status_fields = [
    "pmid",
    "study",
    "source_record_checked",
    "full_text_verified",
    "attrition_handled_appropriately",
    "prespecified_primary_outcome_verified",
    "risk_of_bias_overall",
    "review_status",
]

available_fields = [c for c in status_fields if c in df.columns]
snapshot_path = RESULTS / "predian_appraisal_status_snapshot.csv"
study_rows[available_fields].to_csv(snapshot_path, index=False)

print("PREDIAN SOURCE-VERIFICATION WORKFLOW")
print("=" * 45)
print("Study:", study.get("study", "Not recorded"))
print("PMID:", pmid)
print("Existing appraisal preserved:", APPRAISAL)
print("Verification template:", template_path)
print("Status snapshot:", snapshot_path)
print()
print("Tasks awaiting source-level verification:")
display(template[["priority", "domain", "question"]])

print(
    "\nImportant: the template does not mark any finding as verified. "
    "Enter findings only after checking the original source."
)

PREDIAN SOURCE-VERIFICATION WORKFLOW
Study: PREDIAN final randomized-trial results
PMID: 24970885
Existing appraisal preserved: /kaggle/working/kidneyshield/results/kidneyshield_critical_appraisal.csv
Verification template: /kaggle/working/kidneyshield/results/predian_source_verification_template.csv
Status snapshot: /kaggle/working/kidneyshield/results/predian_appraisal_status_snapshot.csv

Tasks awaiting source-level verification:


,priority,domain,question
0,1,Participant flow,How many randomized participants completed fol...
1,1,Analysis population,"Was the primary analysis intention-to-treat, p..."
2,1,Missing data,How many primary-outcome measurements were mis...
3,1,Protocol and registration,Can the trial registration or protocol confirm...
4,2,Outcome estimates,"Are the eGFR estimate, confidence interval, un..."
5,2,Albuminuria,"What were the exact group estimates, confidenc..."
6,2,Safety,Were serious adverse events and adverse-event ...
7,2,Applicability,How closely does this study population match t...



Important: the template does not mark any finding as verified. Enter findings only after checking the original source.
